# Supplementary tables for the KASPAH data descriptor

| Table | Contents | Rows |
| --- | --- | --- |
| S1 | Per-run ONT sequencing metadata | 31 |
| S2 | Sample manifest | 316 |
| S3 | Software versions | — |
| S4 | Internal-QC adjudications | 28 |
| S5 | QC verdicts and assembly routing | 316 |
| S6 | Typing from all four sources | 294 |
| S7 | Per-isolate read statistics | 294 |
| S8 | Per-assembly statistics | 294 |
| Table 1 | Composition and assembly statistics per species | 5 |

Sections are written in the order the data allows, not in table order: assembly_statistics.tsv is built
before read_statistics.tsv because read depth is bases over assembly length.

In [1]:
import json
import re
from pathlib import Path

import pandas as pd

TABLES = Path("../pipeline_tables")   # the stage summary tables this repo ships
CURATED = TABLES / "curated"          # the hand-maintained inputs (S1 devices, S3, S4, ...)
OUT = Path("supplementary")           # the tables this notebook writes
OUT.mkdir(exist_ok=True)

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
print(f"pandas {pd.__version__}")


pandas       : 3.0.5


## Helpers

`check()` records an assertion instead of raising. `write_table()` is the only way a table
reaches disk — it enforces the global conventions in one place so no individual section can
quietly break them: TSV, Unix line endings, empty string for missing, sorted by isolate then
library, a column-definitions sidecar, and a provenance file naming every source with its
SHA-256.

The FASTA and assembly-metric helpers live here too, rather than in the section that first
needs them: Tables S5 and S8 both measure contigs, and one `parse_fasta` means they cannot
disagree about what a contig is.

In [2]:
CHECKS = []
WRITTEN = []


def check(table, label, ok, detail=""):
    """Record an assertion result. Never raises; the run ends with a report."""
    CHECKS.append({"table": table, "check": label, "ok": bool(ok), "detail": str(detail)})
    mark = "  ok  " if ok else " FAIL "
    print(f"[{mark}] {table:<6} {label}" + (f"  — {detail}" if detail else ""))
    return bool(ok)


def _render(value):
    """Empty string for every flavour of missing. Never NA, NaN or None."""
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return ""
    if value is pd.NA or (not isinstance(value, (list, tuple, set)) and pd.isna(value)):
        return ""
    if isinstance(value, bool):
        return "yes" if value else "no"
    return value


def write_table(df, name, column_defs, sources, sort_by=None):
    """Write one supplementary table, its column definitions and its provenance sidecar."""
    df = df.copy()

    if sort_by is None:
        sort_by = [c for c in ("isolate", "library") if c in df.columns]
    if sort_by:
        df = df.sort_values(sort_by, kind="stable").reset_index(drop=True)

    # Every defined column must exist; anything else in the frame is a working column and is
    # dropped here rather than published.
    missing = [c for c in column_defs if c not in df.columns]
    if missing:
        raise ValueError(f"{name}: column definitions name columns the frame does not have: {missing}")
    df = df[list(column_defs)]

    for col in df.columns:
        df[col] = df[col].map(_render)

    df.to_csv(OUT / f"{name}.tsv", sep="\t", index=False, lineterminator="\n")
    WRITTEN.append(f"{name}.tsv")
    print(f"\n{name}.tsv — {len(df)} rows × {len(df.columns)} columns "
          f"from {', '.join(sources)}")
    return df


def yn(series):
    return series.map(lambda v: "yes" if bool(v) else "no")


def normalise_token(token):
    """`KLE02:1` and `KLE27:NB11` both become `RUN:NBnn`, zero-padded.

    The raw manifest spellings are inconsistent, and matching them verbatim against Dorado's
    output filenames failed silently: `KLE27:NB11` and `KLE31:NB23` resolved to zero files,
    and `KLE02:11` matched 49 files through the `114` in the kit name. The published tokens
    are the normalised form.
    """
    run, _, barcode = token.strip().partition(":")
    digits = re.sub(r"\D", "", barcode)
    return f"{run}:NB{int(digits):02d}" if digits else run


CHROMOSOME_MIN_LEN = 4_000_000


def parse_fasta(path):
    """Yield (header, sequence) pairs."""
    header, chunks = None, []
    with open(path) as fh:
        for line in fh:
            if line.startswith(">"):
                if header is not None:
                    yield header, "".join(chunks)
                header, chunks = line[1:].strip(), []
            else:
                chunks.append(line.strip())
    if header is not None:
        yield header, "".join(chunks)


def gc_percent(seq):
    seq = seq.upper()
    acgt = sum(seq.count(b) for b in "ACGT")
    return 100.0 * (seq.count("G") + seq.count("C")) / acgt if acgt else float("nan")


def n50(lengths):
    lengths = sorted(lengths, reverse=True)
    half, run = sum(lengths) / 2, 0
    for length in lengths:
        run += length
        if run >= half:
            return length
    return 0


def collapse_circular(values):
    values = set(values)
    if "unknown" in values:
        return "unknown"
    return "yes" if values == {"yes"} else "no"

## Sources

Every input the notebook reads, in one place, so the provenance sidecars name real paths and
a missing input fails here rather than three sections later.

In [3]:
SRC = {
    "seq_meta": TABLES / "sequencing_metadata.csv",
    "run_hardware": TABLES / "run_hardware.tsv",
    "manifest": TABLES / "sample_manifest.csv",
    "ont_stats_v1": TABLES / "ont_reads.stats.tsv",
    "ont_stats_v2": TABLES / "ont_reads.stats_v2.tsv",
    "asm_stats": TABLES / "assembly_stats.tsv",
    "internal_qc": TABLES / "internal_qc_summary.tsv",
    "verification": TABLES / "verification_summary.tsv",
    "mapping_summary": TABLES / "mapping_summary.tsv",
    "kraken2_summary": TABLES / "kraken2_summary.tsv",
    "kraken2_reports": "04_verification/taxonomy/output",
    "decontam_targets": TABLES / "decontam_targets.tsv",
    "final_qc": TABLES / "final_qc_summary.tsv",
    "checkm2": TABLES / "checkm2_quality_report.tsv",
    "assemblies": "10_final_assemblies/assemblies",
    "roster": TABLES / "KASPAH_QC_outcome_v2.tsv",
    "collection": TABLES / "KASPAH_collection_nr_170719.tsv",
    "cur_s1_overrides": TABLES / "curated" / "s1_run_overrides.tsv",
    "cur_s1_devices": TABLES / "curated" / "s1_run_devices.tsv",
    "cur_s2_collection": TABLES / "curated" / "s2_source_collection.tsv",
    "cur_s3_software": TABLES / "curated" / "s3_software.tsv",
    "cur_s4": TABLES / "curated" / "s4_adjudications.tsv",
    "cur_expect": TABLES / "curated" / "manuscript_expectations.tsv",
}

absent = [f"{k} -> {v}" for k, v in SRC.items() if not Path(v).exists()]
if absent:
    raise SystemExit("missing inputs:\n  " + "\n  ".join(absent))
print(f"{len(SRC)} inputs present")

23 inputs present


In [4]:
manifest = pd.read_csv(SRC["manifest"])
roster = pd.read_csv(SRC["roster"], sep="\t")
final_qc = pd.read_csv(SRC["final_qc"], sep="\t")
internal_qc = pd.read_csv(SRC["internal_qc"], sep="\t")
seq_meta = pd.read_csv(SRC["seq_meta"])
run_hw = pd.read_csv(SRC["run_hardware"], sep="\t")

print(f"manifest {manifest.shape} · roster {roster.shape} · final_qc {final_qc.shape} "
      f"· internal_qc {internal_qc.shape} · seq_meta {seq_meta.shape} · run_hw {run_hw.shape}")

manifest (316, 10) · roster (316, 21) · final_qc (294, 44) · internal_qc (316, 26) · seq_meta (31, 4) · run_hw (31, 12)


### Run tokens

The manifest records which ONT run and barcode each library came off, `|`-separated where a
library was topped up on a later run. Two of the runs it names — **KLE08 and KLE10** — never
existed on disk: no POD5, no demultiplexed output, no row in `sequencing_metadata.csv`. They
are numbering gaps that the manifest still carries.

Every library naming KLE08 or KLE10 also names a later run, so no library loses its
provenance when those tokens are dropped. Publishing them would say reads came off a run that
produced none, so they are removed here and the affected libraries listed.

In [5]:
S1_RUNS = set(seq_meta.run_id)
manifest["tokens_raw"] = manifest.ont_barcode_map.fillna("").str.split("|")
manifest["tokens"] = manifest.tokens_raw.map(
    lambda ts: [normalise_token(t) for t in ts if t.strip()]
)

phantom = sorted({t.split(":")[0] for ts in manifest.tokens for t in ts} - S1_RUNS)
affected = manifest[manifest.tokens.map(lambda ts: any(t.split(":")[0] in phantom for t in ts))]
print(f"run IDs in the manifest with no run in sequencing_metadata.csv: {phantom}")
print(f"libraries naming one: {len(affected)}")
print(f"of those, libraries left with no token at all after dropping them: "
      f"{sum(1 for ts in affected.tokens if not [t for t in ts if t.split(':')[0] in S1_RUNS])}")

manifest["tokens"] = manifest.tokens.map(
    lambda ts: [t for t in ts if t.split(":")[0] in S1_RUNS]
)
manifest["ont_run_barcode"] = manifest.tokens.map(";".join)

run IDs in the manifest with no run in sequencing_metadata.csv: ['KLE08', 'KLE10']
libraries naming one: 23
of those, libraries left with no token at all after dropping them: 0


## S1 — Per-run ONT sequencing metadata

`sequencing_metadata.csv` carries only what basecalling needed. The flow cell, the instrument
and the sampling rate come from the POD5 `RunInfo` headers, extracted by
`11_manuscript/scan_pod5_runinfo.py` into `../pipeline_tables/run_hardware.tsv`.

**The KLE23 kit is the reason this section reads an overrides file.** Its POD5 header says
`SQK-LSK114-XL`; the wet-lab sheets say `SQK-NBD114-24`. Building the column from the headers
reintroduces the error, so the header value is overridden from
`curated/s1_run_overrides.tsv` and the override is logged when it fires.

Two things the earlier table spec got wrong, both settled by the headers:

- **The instrument is not uniformly a MinION Mk1C.** 24 runs were sequenced on a Mk1B driven
  by a host computer (position `MN24810`) and 5 on a Mk1C (position `MINION1C`).
- **KLE02's flow cell is FLO-MIN112, not FLO-MIN114** — consistent with its `SQK-NBD112-96`
  kit, which is the one kit exception in the cohort. It is the only R10.4 run of the 31.

In [6]:
overrides = pd.read_csv(SRC["cur_s1_overrides"], sep="\t")
devices = pd.read_csv(SRC["cur_s1_devices"], sep="\t", comment="#")

s1 = seq_meta.merge(run_hw, on="run_id", how="left", validate="1:1")
check("S1", "every run in sequencing_metadata.csv has POD5 hardware metadata",
      s1.flow_cell_id.notna().all(),
      f"{s1.flow_cell_id.isna().sum()} without")

# kit: start from the header, then apply the curated override and say so
s1["kit"] = s1.sequencing_kit_header
for row in overrides.itertuples():
    hit = s1.run_id == row.run_id
    before = s1.loc[hit, row.column].iloc[0]
    s1.loc[hit, row.column] = row.value
    print(f"OVERRIDE APPLIED  {row.run_id}.{row.column}: {before!r} -> {row.value!r}\n"
          f"                  {row.reason}")

# the corrected CSV is an independent record of the same decision; they must agree
kit_csv = seq_meta.set_index("run_id").kit
mismatch = [r for r in s1.itertuples() if r.kit != kit_csv[r.run_id]]
check("S1", "overridden kit agrees with the corrected sequencing_metadata.csv",
      not mismatch, f"{[m.run_id for m in mismatch]}")

s1["device"] = s1.device_type.map(devices.set_index("device_type").device)
serial = devices.set_index("device_type").instrument_serial
s1["device"] = [
    f"{d} {serial[t]}" if isinstance(serial.get(t), str) and serial[t].strip() else d
    for d, t in zip(s1.device, s1.device_type)
]

s1["basecalling_model"] = s1.model
s1["translocation_speed_bps"] = s1.model.str.extract(r"_(\d+)bps_").astype(int)

tokens_by_run = {}
for row in manifest.itertuples():
    for tok in row.tokens:
        tokens_by_run.setdefault(tok.split(":")[0], []).append((tok, row.strain_id))
s1["n_barcodes"] = s1.run_id.map(lambda r: len({t for t, _ in tokens_by_run.get(r, [])}))

# 17 libraries were topped up on a second run, so the barcode positions outnumber the
# libraries. Naming them by library would assert something false.
multi_run = sorted(manifest.loc[manifest.tokens.map(len) > 1, "strain_id"])
print(f"barcode positions {sum(s1.n_barcodes)} = 316 libraries + {len(multi_run)} sequenced "
      f"across two runs:\n  {', '.join(multi_run)}")

S1_COLS = {
    "run_id": "ONT run identifier as used throughout the repository.",
    "device": "Sequencing instrument model, from the POD5 tracking_id block: MinION Mk1B for 24 runs, MinION Mk1C for 5 (KLE20, KLE23, KLE25, KLE27, KLE29), GridION GXB04712 for RM01/RM02 (serial from 00_raw_data/RESEQUENCING/README.md). NOT YET CONFIRMED AGAINST THE WET-LAB RECORDS - the KLE23 kit shows these headers are not always right, and this column feeds a manuscript sentence.",
    "flow_cell_id": "Flow cell serial, from the POD5 RunInfo block.",
    "flow_cell_type": "Flow cell product code, from the POD5 RunInfo block.",
    "kit": "Library preparation kit. KLE23 is a curated override of the POD5 header value; see ../pipeline_tables/curated/s1_run_overrides.tsv.",
    "sampling_rate_hz": "Raw signal sampling rate, from the POD5 RunInfo block. 4000 or 5000.",
    "translocation_speed_bps": "DNA translocation speed in bases per second, from the basecalling model name.",
    "basecalling_model": "Dorado super-accuracy model used, selected from the sampling rate.",
    "batch_size": "Dorado basecalling batch size.",
    "n_barcodes": "Distinct barcodes on this run that map to a sequenced library. These sum to 333 across the 31 runs, not 316: 17 libraries were sequenced across two runs and so occupy two barcode positions. Which 17 is derivable from sample_manifest.tsv - the rows whose ont_run_barcode holds more than one token.",
}

S1 = write_table(s1, "Table_S1_ont_run_metadata", S1_COLS,
                 [SRC["seq_meta"], SRC["run_hardware"], SRC["manifest"],
                  SRC["cur_s1_overrides"], SRC["cur_s1_devices"]],
                 sort_by=["run_id"])

[  ok  ] S1     every run in sequencing_metadata.csv has POD5 hardware metadata  — 0 without
OVERRIDE APPLIED  KLE23.kit: 'SQK-NBD114-24' -> 'SQK-NBD114-24'
                  POD5 header reads sqk-lsk114-xl; the wet-lab sample sheets establish SQK-NBD114-24. Corrected by hand in ../pipeline_tables/sequencing_metadata.csv before basecalling and recorded in 01_preprocessing/README.md as a critical manual step.
[  ok  ] S1     overridden kit agrees with the corrected sequencing_metadata.csv  — []
barcode positions 333 = 316 libraries + 17 sequenced across two runs:
  INF017, INF019, INF027, INF074, INF092.1, INF096, INF138, INF186, INF216, INF284, INF349, INF360, KSB1_2J, KSB1_3B, KSB1_3C, KSB1_3F, KSB1_3G

Table_S1_ont_run_metadata.tsv — 31 rows × 10 columns from ../pipeline_tables/sequencing_metadata.csv, ../pipeline_tables/run_hardware.tsv, ../pipeline_tables/sample_manifest.csv, ../pipeline_tables/curated/s1_run_overrides.tsv, ../pipeline_tables/curated/s1_run_devices.tsv


In [7]:
check("S1", "31 rows", len(S1) == 31, len(S1))
check("S1", "no null in kit, basecalling_model, sampling_rate_hz",
      (S1[["kit", "basecalling_model", "sampling_rate_hz"]] != "").all().all())
check("S1", "sampling_rate_hz in {4000, 5000}",
      set(S1.sampling_rate_hz.astype(int)) <= {4000, 5000},
      sorted(set(S1.sampling_rate_hz.astype(int))))
rate_model = S1.assign(rate=S1.sampling_rate_hz.astype(int))
bad = rate_model[
    ((rate_model.rate == 4000) & ~rate_model.basecalling_model.str.contains("v4.0.0"))
    | ((rate_model.rate == 5000) & ~rate_model.basecalling_model.str.contains("v4.3.0"))
]
check("S1", "sampling rate consistent with model version (4 kHz=v4.0.0, 5 kHz=v4.3.0)",
      bad.empty, list(bad.run_id))
check("S1", "KLE23 kit reads SQK-NBD114-24",
      S1.loc[S1.run_id == "KLE23", "kit"].iloc[0] == "SQK-NBD114-24",
      S1.loc[S1.run_id == "KLE23", "kit"].iloc[0])
check("S1", "kit is SQK-NBD114-24 everywhere except KLE02",
      set(S1[S1.run_id != "KLE02"].kit) == {"SQK-NBD114-24"}
      and S1.loc[S1.run_id == "KLE02", "kit"].iloc[0] == "SQK-NBD112-96")
check("S1", "n_barcodes sums to 333 = 316 libraries + 17 sequenced across two runs",
      S1.n_barcodes.astype(int).sum() == 316 + 17 == 333,
      f"{S1.n_barcodes.astype(int).sum()} barcode positions, {len(multi_run)} multi-run libraries")
check("S1", "every library has at least one run token",
      manifest.tokens.map(len).gt(0).all(),
      f"{(manifest.tokens.map(len) == 0).sum()} without")

S1.head(4)

[  ok  ] S1     31 rows  — 31
[  ok  ] S1     no null in kit, basecalling_model, sampling_rate_hz
[  ok  ] S1     sampling_rate_hz in {4000, 5000}  — [4000, 5000]
[  ok  ] S1     sampling rate consistent with model version (4 kHz=v4.0.0, 5 kHz=v4.3.0)
[  ok  ] S1     KLE23 kit reads SQK-NBD114-24  — SQK-NBD114-24
[  ok  ] S1     kit is SQK-NBD114-24 everywhere except KLE02
[  ok  ] S1     n_barcodes sums to 333 = 316 libraries + 17 sequenced across two runs  — 333 barcode positions, 17 multi-run libraries
[  ok  ] S1     every library has at least one run token  — 0 without


,run_id,device,flow_cell_id,flow_cell_type,kit,sampling_rate_hz,translocation_speed_bps,basecalling_model,batch_size,n_barcodes
0,KLE02,MinION Mk1B,FAU75969,FLO-MIN112,SQK-NBD112-96,4000,260,dna_r10.4.1_e8.2_260bps_sup@v4.0.0,2000,11
1,KLE03,MinION Mk1B,FAV99435,FLO-MIN114,SQK-NBD114-24,4000,260,dna_r10.4.1_e8.2_260bps_sup@v4.0.0,2000,12
2,KLE04,MinION Mk1B,FAV99315,FLO-MIN114,SQK-NBD114-24,4000,260,dna_r10.4.1_e8.2_260bps_sup@v4.0.0,2000,12
3,KLE05,MinION Mk1B,FAV99302,FLO-MIN114,SQK-NBD114-24,4000,260,dna_r10.4.1_e8.2_260bps_sup@v4.0.0,2000,11


## S2 — Sample manifest

All 316 libraries, whether or not they were released. `source_collection` maps each isolate
to the study it came from via `../pipeline_tables/KASPAH_collection_nr_170719.tsv`, using the labels in
`curated/s2_source_collection.tsv`.

`illumina_accession` is deliberately **empty for the 22 libraries sequenced in-house** — the
manifest holds a local sample name (`INF034_S1`) there, not an accession, and publishing it
in an accession column would invite readers to look it up. Those cells get real values once
the reads are deposited.

In [8]:
collection = pd.read_csv(SRC["collection"], sep="\t", dtype=str)
coll_labels = pd.read_csv(SRC["cur_s2_collection"], sep="\t")
ref_to_label = dict(zip(coll_labels.reference, coll_labels.source_collection))

collection["isolate_key"] = collection["sample ID"].str.replace(r"\.\d+$", "", regex=True)
ref_by_isolate = (collection.drop_duplicates("isolate_key")
                  .set_index("isolate_key").Reference.map(ref_to_label))

roster_by_lib = roster.set_index("sample")

EXCLUSION = {
    "FAIL_VARIANT": "duplicate_colony_pick",
    "DROP_VARIANT": "duplicate_colony_pick",
    "DROP_TYPING_DISCORDANT": "platform_k_locus_discordance",
    "DROP_WRONG_ISOLATE": "library_not_labelled_isolate",
    "DROP_INTERNAL_MISMATCH": "platform_species_discordance",
}

s2 = pd.DataFrame({
    "isolate": manifest.kaspah_id,
    "library": manifest.strain_id,
})
s2["colony_pick"] = manifest.strain_id.str.extract(r"(\.\d)(?:_v2)?$")[0].fillna("")
s2["resequenced"] = yn(manifest.data_version == "v2")
s2["ont_run_barcode"] = manifest.ont_run_barcode
s2["illumina_source"] = manifest.illumina_source.map(
    {"SRA": "archived", "131_RM": "generated_here"})
s2["illumina_accession"] = manifest.illumina_accession.where(
    manifest.illumina_source == "SRA", "").fillna("")
s2["source_collection"] = manifest.kaspah_id.map(ref_by_isolate).fillna("")
s2["released"] = manifest.strain_id.map(roster_by_lib.qc_outcome).map(
    {"KEEP": "yes", "DROP": "no"})
s2["exclusion_reason"] = [
    "" if rel == "yes" else EXCLUSION.get(roster_by_lib.qc_decision.get(lib), "")
    for rel, lib in zip(s2.released, s2.library)
]

S2_COLS = {
    "isolate": "Biological sample. One released genome per isolate.",
    "library": "One sequenced preparation of an isolate. The `_v2` suffix marks the 131_RM resequencing batch.",
    "colony_pick": "`.1` or `.2` where the isolate was plated and two distinct colony morphologies were picked and prepared separately. Empty where the isolate was picked once. These are not technical replicates.",
    "resequenced": "`yes` where the library belongs to the 131_RM batch, re-cultured and resequenced on both platforms after an earlier QC pass flagged it.",
    "ont_run_barcode": "Semicolon-separated RUN:BARCODE tokens, zero-padded and normalised; more than one where the library was topped up on a later run. Tokens naming KLE08 or KLE10 are omitted: those runs never existed on disk and produced no reads.",
    "illumina_source": "`archived` where the Illumina reads are an existing public accession; `generated_here` where they were sequenced for this study.",
    "illumina_accession": "ENA or SRA run accession. Empty for the 22 libraries sequenced here, pending deposition.",
    "source_collection": "The study the isolate was collected in, from ../pipeline_tables/KASPAH_collection_nr_170719.tsv.",
    "released": "`yes` for the 294 libraries whose assembly is in the released set.",
    "exclusion_reason": "Why a library was not released. Empty where released. `duplicate_colony_pick` means the isolate is released via its other colony pick; the other three values each cost the collection an isolate.",
}

S2 = write_table(s2, "sample_manifest", S2_COLS,
                 [SRC["manifest"], SRC["roster"], SRC["final_qc"], SRC["collection"],
                  SRC["cur_s2_collection"]])


Table_S2_sample_manifest.tsv — 316 rows × 10 columns from ../pipeline_tables/sample_manifest.csv, ../pipeline_tables/KASPAH_QC_outcome_v2.tsv, ../pipeline_tables/final_qc_summary.tsv, ../pipeline_tables/KASPAH_collection_nr_170719.tsv, ../pipeline_tables/curated/s2_source_collection.tsv


In [9]:
check("S2", "316 rows", len(S2) == 316, len(S2))
check("S2", "298 distinct isolates", S2.isolate.nunique() == 298, S2.isolate.nunique())
two = S2.groupby("isolate").size()
check("S2", "exactly 18 isolates with two rows", (two == 2).sum() == 18, (two == 2).sum())
check("S2", "no isolate has more than two rows", two.max() == 2, two.max())
check("S2", "released == yes for exactly 294", (S2.released == "yes").sum() == 294,
      (S2.released == "yes").sum())
rel_per_iso = S2[S2.released == "yes"].groupby("isolate").size()
check("S2", "exactly one released library per released isolate",
      (rel_per_iso == 1).all() and len(rel_per_iso) == 294, len(rel_per_iso))
check("S2", "exclusion_reason non-empty for exactly 22",
      (S2.exclusion_reason != "").sum() == 22, (S2.exclusion_reason != "").sum())
check("S2", "18 of those are duplicate_colony_pick",
      (S2.exclusion_reason == "duplicate_colony_pick").sum() == 18,
      S2[S2.exclusion_reason != ""].exclusion_reason.value_counts().to_dict())
check("S2", "illumina_source splits 294 archived / 22 generated_here",
      S2.illumina_source.value_counts().to_dict() == {"archived": 294, "generated_here": 22},
      S2.illumina_source.value_counts().to_dict())
acc = S2.loc[S2.illumina_accession != "", "illumina_accession"]
check("S2", "278 distinct non-empty accessions", acc.nunique() == 278, acc.nunique())
check("S2", "16 accessions appear twice",
      (acc.value_counts() == 2).sum() == 16 and acc.value_counts().max() == 2,
      f"{(acc.value_counts() == 2).sum()} twice, max {acc.value_counts().max()}")
tok_runs = {t.split(":")[0] for s in S2.ont_run_barcode if s for t in s.split(";")}
check("S2", "every run token resolves to a run in S1", tok_runs <= set(S1.run_id),
      sorted(tok_runs - set(S1.run_id)))
check("S2", "source_collection populated for every row",
      (S2.source_collection != "").all(),
      f"{(S2.source_collection == '').sum()} empty: {sorted(S2[S2.source_collection == ''].isolate)}")

print()
print(S2.source_collection.value_counts().to_string())

[  ok  ] S2     316 rows  — 316
[  ok  ] S2     298 distinct isolates  — 298
[  ok  ] S2     exactly 18 isolates with two rows  — 18
[  ok  ] S2     no isolate has more than two rows  — 2
[  ok  ] S2     released == yes for exactly 294  — 294
[  ok  ] S2     exactly one released library per released isolate  — 294
[  ok  ] S2     exclusion_reason non-empty for exactly 22  — 22
[  ok  ] S2     18 of those are duplicate_colony_pick  — {'duplicate_colony_pick': 18, 'platform_k_locus_discordance': 2, 'library_not_labelled_isolate': 1, 'platform_species_discordance': 1}
[  ok  ] S2     illumina_source splits 294 archived / 22 generated_here  — {'archived': 294, 'generated_here': 22}
[  ok  ] S2     278 distinct non-empty accessions  — 278
[  ok  ] S2     16 accessions appear twice  — 16 twice, max 2
[  ok  ] S2     every run token resolves to a run in S1
[  ok  ] S2     source_collection populated for every row  — 0 empty: []

source_collection
hospital-wide infection    237
ICU carriage   

## S3 — Software versions

A pass-through of a curated list; nothing is computed. Two values were corrected against the
live environments during preparation and the curated file says so in `notes`: **Kraken2 2.1.7**
(`MASTER_DOC.md` §6 records 2.17.1, which is not a released version) and **seqkit 2.11.0**
(recorded as 2.3.1). **fastp is 0.23.4**, the module `bin/05_run_qc_assembly.sh` loads, and
the build asserts that against the script rather than against the per-library JSON reports —
some of those were carried over from the v1 tree and report a version this pipeline never ran.

The published table is three columns. The curated input carries two more —
`version_provenance` (`logged_at_runtime` / `verified_post_hoc` / `reconstructed`) and
`notes` — which are checked here but not shipped: they are the audit trail for how each
version was established, not something a reader of the supplement needs. The coverage is
uneven, and if Methods gains a sentence about version provenance this is where the evidence
for it lives.

The analysis-environment table that used to ship as S2b is no longer part of the supplement.
Its curated input, `../pipeline_tables/curated/s2_analysis_env.tsv`, is left in place unreferenced,
so nothing has to be rebuilt if it is wanted again.

In [10]:
s3_software = pd.read_csv(SRC["cur_s3_software"], sep="\t").fillna("")

# fastp is a curated constant, not a parse. `bin/05_run_qc_assembly.sh` loads the module
# `fastp/0.23.4`, and that is the version this pipeline ran. Deriving the value from the
# per-library JSON reports picks up whatever wrote each report — including reports carried
# over from the v1 tree — so it is deliberately not derived here.
FASTP_PIN = "fastp/0.23.4"
qc_script = Path("bin/05_run_qc_assembly.sh").read_text()
curated_fastp = s3_software.loc[s3_software.tool == "fastp", "version"].iloc[0]
check("S3", "the curated fastp version matches the module pinned in bin/05_run_qc_assembly.sh",
      FASTP_PIN in qc_script and curated_fastp == FASTP_PIN.split("/")[1],
      f"script pins {FASTP_PIN}, table says {curated_fastp}")

# version_provenance must agree with the runtime log where one exists
logged = pd.read_csv(TABLES / "tool_versions.tsv", sep="\t")
logged_tools = {t.split("_")[0].lower() for t in logged.tool}
claimed = {t.lower() for t in s3_software.loc[
    s3_software.version_provenance == "logged_at_runtime", "tool"]}
check("S3", "every tool claiming logged_at_runtime appears in provenance/tool_versions.tsv",
      claimed <= logged_tools, sorted(claimed - logged_tools))
check("S3", "no tool in the runtime log is marked reconstructed",
      not (logged_tools & {t.lower() for t in s3_software.loc[
          s3_software.version_provenance == "reconstructed", "tool"]}),
      sorted(logged_tools & {t.lower() for t in s3_software.loc[
          s3_software.version_provenance == "reconstructed", "tool"]}))

S3_COLS = {
    "tool": "Software name as cited in Methods.",
    "version": "Version used.",
    "role_in_workflow": "What the tool did in this study.",
}

S3 = write_table(s3_software, "Table_S2_software_versions", S3_COLS,
                 [SRC["cur_s3_software"]], sort_by=["tool"])

unresolved = list(S3.loc[S3.version == "confirm", "tool"])
check("S3", "no `confirm` flags remain", not unresolved, unresolved)

[  ok  ] S3     the curated fastp version matches the module pinned in bin/05_run_qc_assembly.sh  — script pins fastp/0.23.4, table says 0.23.4
[  ok  ] S3     every tool claiming logged_at_runtime appears in provenance/tool_versions.tsv
[  ok  ] S3     no tool in the runtime log is marked reconstructed

Table_S3_software_versions.tsv — 27 rows × 3 columns from ../pipeline_tables/curated/s3_software.tsv
[  ok  ] S3     no `confirm` flags remain


True

## S4 — The 28 internal-QC adjudications

The automatic gate is
`asm_n_chromosomes == 1 & mapped_pct >= 95 & kbt_K_locus_concordant == 'YES' &
kraken2_status == 'PASS'`. 288 of 316 libraries satisfied all four with no human involvement.
The 28 that did not were each inspected and assigned one of seven verdicts — a fan-out, not a
binary split.

The metrics are recomputed here; the reasoning is read from `curated/s4_adjudications.tsv`,
because a judgement is not derivable from the numbers that prompted it.

**`mapped_pct` is stale for the four decontaminated libraries.** Illumina was mapped against
the pre-decontamination assemblies. That is correct for internal QC, which ran before
remediation, but the value must not be quoted for those rows, and the column definition says so.

In [11]:
curated_s4 = pd.read_csv(SRC["cur_s4"], sep="\t")

gate = ((internal_qc.asm_n_chromosomes == 1) & (internal_qc.mapped_pct >= 95)
        & (internal_qc.kbt_K_locus_concordant == "YES")
        & (internal_qc.kraken2_status == "PASS"))
adjudicated = internal_qc[~gate].copy()
print(f"gate passed {gate.sum()} · adjudicated {(~gate).sum()}")

PREDICATES = {
    "chromosome_count": adjudicated.asm_n_chromosomes != 1,
    "mapping_rate": adjudicated.mapped_pct < 95,
    "k_locus_concordance": adjudicated.kbt_K_locus_concordant != "YES",
    "kraken2_status": adjudicated.kraken2_status != "PASS",
}
adjudicated["failed_predicates"] = [
    ";".join(name for name, mask in PREDICATES.items() if mask.iloc[i])
    for i in range(len(adjudicated))
]
print({name: int(mask.sum()) for name, mask in PREDICATES.items()})

s4 = adjudicated.rename(columns={"sample": "library"})[[
    "library", "failed_predicates", "asm_n_chromosomes", "mapped_pct",
    "kbt_K_locus_concordant", "kraken2_status",
]].merge(curated_s4, on="library", how="left", validate="1:1")

unjoined = sorted(s4.loc[s4.verdict.isna(), "library"])
check("S4", "every adjudicated library has curated reasoning", not unjoined, unjoined)
stale = sorted(set(curated_s4.library) - set(s4.library))
check("S4", "no curated row is left over", not stale, stale)

s4["isolate"] = s4.library.map(roster_by_lib.isolate)
s4["released"] = s4.library.map(
    roster_by_lib.qc_outcome).map({"KEEP": "yes", "DROP": "no"})

# the verdict recorded in the pipeline and the verdict in the curated file must be the same
pipeline_verdict = adjudicated.set_index("sample").internal_qc_decision
disagree = [lib for lib, v in zip(s4.library, s4.verdict) if pipeline_verdict[lib] != v]
check("S4", "curated verdict matches internal_qc_summary.tsv", not disagree, disagree)

S4_COLS = {
    "library": "One sequenced preparation.",
    "isolate": "Biological sample.",
    "failed_predicates": "Semicolon-separated subset of chromosome_count, mapping_rate, k_locus_concordance, kraken2_status. A library can fail several.",
    "asm_n_chromosomes": "Contigs >= 4 Mb in the Nanopore-only assembly. The gate requires exactly one.",
    "mapped_pct": "Percentage of Illumina reads mapping to the Nanopore-only assembly. STALE FOR THE FOUR DECONTAMINATED LIBRARIES: Illumina was mapped against the pre-decontamination assemblies. Correct for internal QC, which ran first; do not quote it for those rows.",
    "kbt_K_locus_concordant": "Whether the Nanopore-only and Illumina-only assemblies gave the same K locus.",
    "kraken2_status": "Verdict of the Kraken2 contamination screen, at the thresholds pinned in bin/08_run_kraken2.sh and bin/scripts/kraken2_parser.py.",
    "verdict": "One of seven adjudications. The three PASS_* verdicts released the library; REDO_* sent it to remediation; FAIL_VARIANT dropped the losing colony pick.",
    "reasoning": "Why the verdict was reached. Curated; not derivable from the metrics.",
    "outcome": "What actually happened to the library after the verdict.",
    "released": "`yes` where the library's assembly is in the released set.",
}
S4 = write_table(s4, "Table_S3_internal_qc", S4_COLS, [SRC["internal_qc"], SRC["cur_s4"]])

gate passed 288 · adjudicated 28
{'chromosome_count': 8, 'mapping_rate': 12, 'k_locus_concordance': 12, 'kraken2_status': 16}
[  ok  ] S4     every adjudicated library has curated reasoning
[  ok  ] S4     no curated row is left over
[  ok  ] S4     curated verdict matches internal_qc_summary.tsv

Table_S4_internal_qc_adjudications.tsv — 28 rows × 11 columns from ../pipeline_tables/internal_qc_summary.tsv, ../pipeline_tables/curated/s4_adjudications.tsv


In [12]:
check("S4", "28 rows", len(S4) == 28, len(S4))
check("S4", "verdict counts reproduce the project record",
      S4.verdict.value_counts().to_dict() == {
          "FAIL_VARIANT": 10, "PASS_PROVISIONAL_CHECK_EXTERNAL": 4,
          "REDO_DECONTAM_KLEBSIELLA": 4, "PASS_BORDERLINE_MAPPING": 3,
          "REDO_ASSEMBLY_META": 3, "PASS_FALSE_POSITIVE_CONTAM": 2,
          "REDO_DECONTAM_ECOLI": 2},
      S4.verdict.value_counts().to_dict())
check("S4", "every row fails at least one predicate", (S4.failed_predicates != "").all())
check("S4", "failed predicate counts: kraken2 16, mapping 12, k locus 12, chromosome 8",
      {n: int(m.sum()) for n, m in PREDICATES.items()} == {
          "kraken2_status": 16, "mapping_rate": 12,
          "k_locus_concordance": 12, "chromosome_count": 8},
      {n: int(m.sum()) for n, m in PREDICATES.items()})
check("S4", "28 adjudicated + 288 gate-passing = 316",
      len(S4) + int(gate.sum()) == 316, len(S4) + int(gate.sum()))

S4[["library", "verdict", "failed_predicates", "released"]]

[  ok  ] S4     28 rows  — 28
[  ok  ] S4     verdict counts reproduce the project record  — {'FAIL_VARIANT': 10, 'REDO_DECONTAM_KLEBSIELLA': 4, 'PASS_PROVISIONAL_CHECK_EXTERNAL': 4, 'PASS_BORDERLINE_MAPPING': 3, 'REDO_ASSEMBLY_META': 3, 'PASS_FALSE_POSITIVE_CONTAM': 2, 'REDO_DECONTAM_ECOLI': 2}
[  ok  ] S4     every row fails at least one predicate
[  ok  ] S4     failed predicate counts: kraken2 16, mapping 12, k locus 12, chromosome 8  — {'chromosome_count': 8, 'mapping_rate': 12, 'k_locus_concordance': 12, 'kraken2_status': 16}
[  ok  ] S4     28 adjudicated + 288 gate-passing = 316  — 316


,library,verdict,failed_predicates,released
0,INF001.2,FAIL_VARIANT,mapping_rate;k_locus_concordance,no
1,INF050,PASS_BORDERLINE_MAPPING,mapping_rate,yes
2,INF071.2,FAIL_VARIANT,mapping_rate;k_locus_concordance,no
3,INF092.2,FAIL_VARIANT,mapping_rate;k_locus_concordance;kraken2_status,no
4,INF093.2,FAIL_VARIANT,mapping_rate,no
5,INF096,REDO_ASSEMBLY_META,chromosome_count;k_locus_concordance,no
6,INF117.2,FAIL_VARIANT,mapping_rate;k_locus_concordance,no
7,INF184,PASS_FALSE_POSITIVE_CONTAM,kraken2_status,yes
8,INF186,REDO_DECONTAM_KLEBSIELLA,chromosome_count;k_locus_concordance;kraken2_s...,yes
9,INF195,PASS_PROVISIONAL_CHECK_EXTERNAL,kraken2_status,yes


## S5 — QC verdicts and assembly routing, extended to all 316 libraries

Built for the 294 released isolates as a projection of `final_qc_summary.tsv`; the 22
excluded libraries are appended from the roster and the internal-QC summary, so the exclusion
narrative in Methods is checkable against a table rather than only against prose.

For an excluded row the internal verdict is populated, the external verdict is populated
where the library got that far, and `final_qc_decision`, `read_prep`, `assembly_route` and
every hybrid and Δ column are empty — because none of those stages happened to it.

The `d_*` columns are hybrid minus Nanopore-only: what polishing changed.

**Twelve internal-QC measurements are carried here for all 316 libraries** — the mapping
rate, the Kraken2 genus profile and the two platforms' Kleborate calls. They are the
evidence behind `internal_qc_decision`, and Figures 3 and 4 plot them. The point of putting
them in the table is that the figures then read from the same file the text and the
supplement do, and a number cannot move in one place without moving in the others. Nothing
here is recomputed for the figures: the values come from
`../pipeline_tables/verification_summary.tsv` and the per-sample Kraken2 reports it was built
from, and the section checks them back against `mapping_summary.tsv`, `kraken2_summary.tsv`
and `internal_qc_summary.tsv`.

Two of them read differently from the same-named columns in genome_typing.tsv and the definitions say
so. `mapped_pct` is measured against the pre-decontamination assembly for the four
decontaminated libraries, and `ont_species`/`ont_ST`/`ont_K_locus` are the first-pass
Nanopore calls, which is the assembly internal QC judged — genome_typing.tsv reports the assembly each
genome was finally built from. The two disagree for INF186 and INF216, and a cross-table
check names them, so a third cannot appear unnoticed.

`klebsiella_genus_read_fraction` and `top_non_klebsiella_genus_fraction` are **fractions,
0–1**, unlike every `_pct` column in this table. The names say so; the axis labels should
too.

In [13]:
KLEBSIELLA_RELATED = {"Klebsiella", "Raoultella", "unclassified Klebsiella"}


def genus_profile(path):
    """Genus-level shares from one Kraken2 report.

    Returns (Klebsiella %, top non-Klebsiella genus, that genus's %) from the report's own
    rank-G rows, using the same walk and the same exclusion set as
    `bin/scripts/kraken2_parser.py` — so these columns and `kraken2_status` cannot describe
    different readings of the same report. The percentages are the report's: a share of all
    reads, unclassified included.
    """
    klebsiella, top = 0.0, ("", 0.0)
    for line in Path(path).read_text().splitlines():
        fields = line.split("\t")
        if len(fields) < 6:
            continue
        pct, rank, name = float(fields[0]), fields[3].strip(), fields[5].strip()
        if rank != "G":
            continue
        if name == "Klebsiella":
            klebsiella = pct
        elif name not in KLEBSIELLA_RELATED and pct > top[1]:
            top = (name, pct)
    return klebsiella, top[0], top[1]


verification = pd.read_csv(SRC["verification"], sep="\t")

# `../pipeline_tables/internal_qc_summary.tsv` is this file plus the verdict column. This table
# takes its verdict from one and its metrics from the other, so they have to be the same QC
# pass rather than two vintages of it.
shared = [c for c in verification.columns if c in internal_qc.columns]
check("S5", "04_verification and 05_internal_qc agree on all 25 shared columns",
      len(shared) == 25
      and verification.sort_values("sample")[shared].reset_index(drop=True).equals(
          internal_qc.sort_values("sample")[shared].reset_index(drop=True)),
      f"{len(shared)} shared columns")

genus_rows = []
for lib in verification["sample"]:
    report = Path(SRC["kraken2_reports"]) / lib / f"{lib}_kraken2.report"
    if not report.exists():
        raise FileNotFoundError(f"no Kraken2 report for {lib}: {report}")
    klebsiella_pct, genus, genus_pct = genus_profile(report)
    genus_rows.append({
        "library": lib,
        "klebsiella_genus_read_fraction": round(klebsiella_pct / 100, 4),
        "top_non_klebsiella_genus": genus,
        "top_non_klebsiella_genus_fraction": round(genus_pct / 100, 4),
    })

verification_metrics = pd.DataFrame({
    "library": verification["sample"],
    "mapped_pct": verification.mapped_pct,
    "properly_paired_pct": verification.properly_paired_pct,
    "kraken2_status": verification.kraken2_status,
    "ont_species": verification.kbt_ont_species,
    "ont_ST": verification.kbt_ont_ST,
    "ont_K_locus": verification.kbt_ont_K_locus,
    "illumina_species": verification.kbt_illumina_species,
    "illumina_ST": verification.kbt_illumina_ST,
    "illumina_K_locus": verification.kbt_illumina_K_locus,
}).merge(pd.DataFrame(genus_rows), on="library", validate="1:1")

# Each column is checked back against the file it was summarised from, so "sourced from
# 04_verification" is a claim the notebook proves rather than one it makes.
mapping = pd.read_csv(SRC["mapping_summary"], sep="\t").set_index("sample")
metrics_by_lib = verification_metrics.set_index("library")
for col in ("mapped_pct", "properly_paired_pct"):
    same = metrics_by_lib[col].round(2).eq(mapping[col].round(2).reindex(metrics_by_lib.index))
    check("S5", f"{col} reproduces mapping_summary.tsv", same.all(),
          sorted(metrics_by_lib.index[~same]))

k2 = pd.read_csv(SRC["kraken2_summary"], sep="\t").set_index("sample")
same = (metrics_by_lib.klebsiella_genus_read_fraction * 100).round(2).eq(
    k2.klebsiella_pct.round(2).reindex(metrics_by_lib.index))
check("S5", "klebsiella_genus_read_fraction reproduces kraken2_summary.tsv",
      same.all(), sorted(metrics_by_lib.index[~same]))
flagged = set(metrics_by_lib.index[metrics_by_lib.top_non_klebsiella_genus_fraction >= 0.05])
named = set(k2.contaminants_detail.dropna().index)
check("S5", "the libraries with a top non-Klebsiella genus >= 5% are exactly the 9 kraken2_summary.tsv flags",
      flagged == named and len(flagged) == 9, sorted(flagged ^ named))

# A figure over 314 libraries that claims 316 is the error this refuses to let through.
absent = {
    col: sorted(verification_metrics.loc[
        verification_metrics[col].isna()
        | verification_metrics[col].astype(str).str.strip().eq(""), "library"])
    for col in verification_metrics.columns
}
absent = {col: libs for col, libs in absent.items() if libs}
check("S5", "all 316 libraries carry all 12 verification metrics",
      len(verification_metrics) == 316 and not absent, absent)
if absent:
    raise ValueError(f"verification metrics missing values: {absent}")
print(f"verification metrics: {len(verification_metrics)} libraries x "
      f"{len(verification_metrics.columns) - 1} columns")
print(verification_metrics.top_non_klebsiella_genus.value_counts().head(6).to_string())

[  ok  ] S5     04_verification and 05_internal_qc agree on all 25 shared columns  — 25 shared columns
[  ok  ] S5     mapped_pct reproduces mapping_summary.tsv
[  ok  ] S5     properly_paired_pct reproduces mapping_summary.tsv
[  ok  ] S5     klebsiella_genus_read_fraction reproduces kraken2_summary.tsv
[  ok  ] S5     the libraries with a top non-Klebsiella genus >= 5% are exactly the 9 kraken2_summary.tsv flags
[  ok  ] S5     all 316 libraries carry all 12 verification metrics
verification metrics: 316 libraries x 12 columns
top_non_klebsiella_genus
Escherichia     153
Homo             80
Enterobacter     49
Citrobacter      10
Salmonella        6
Serratia          4


In [14]:
S5_KEEP = [
    "isolate", "library", "morphology", "resequenced", "internal_qc_decision",
    "external_qc_decision", "external_mismatch_reason", "final_qc_decision", "read_prep",
    "assembly_route", "hybrid_contigs", "hybrid_total_len", "hybrid_chromosome_len",
    "ont_contigs", "ont_total_len", "ont_chromosome_len", "d_contigs", "d_total_len",
    "d_chromosome_len",
]
s5_rel = final_qc[S5_KEEP].copy()
s5_rel["colony_pick"] = s5_rel.morphology.map(
    lambda m: f".{int(m)}" if pd.notna(m) else "")
s5_rel = s5_rel.drop(columns=["morphology"])
s5_rel["resequenced"] = yn(s5_rel.resequenced.astype(bool))
s5_rel["released"] = "yes"
s5_rel["exclusion_reason"] = ""

excluded = S2[S2.released == "no"][["isolate", "library", "colony_pick", "resequenced",
                                    "exclusion_reason"]].copy()
try:
    ext_qc = pd.read_csv(TABLES / "external_qc_summary.tsv", sep="\t")
    ext_by_lib = ext_qc.set_index("sample")
except FileNotFoundError:
    ext_by_lib = pd.DataFrame()

excluded["internal_qc_decision"] = excluded.library.map(
    internal_qc.set_index("sample").internal_qc_decision)
excluded["external_qc_decision"] = excluded.library.map(
    ext_by_lib.external_qc_decision if len(ext_by_lib) else {}).fillna("")
excluded["external_mismatch_reason"] = excluded.library.map(
    ext_by_lib.mismatch_reason if len(ext_by_lib) else {}).fillna("")
for col in ["final_qc_decision", "read_prep", "assembly_route", "hybrid_contigs",
            "hybrid_total_len", "hybrid_chromosome_len", "d_contigs", "d_total_len",
            "d_chromosome_len"]:
    excluded[col] = ""

# Every excluded library has a Nanopore assembly — that is what it was judged on — so the ONT
# columns are populated, and the exclusion narrative becomes checkable.
#
# It has to be the **best assembly the library reached**, not the first-pass one. INF096 and
# INF294 went through Autocycler before being dropped, and internal QC's first-pass figures
# for them are 30 and 23 contigs against the 2 and 3 the consensus produced. Quoting the
# first pass would say they were excluded for assembling badly, which is the opposite of what
# happened: both assembled acceptably and were dropped because ONT and Illumina typed to
# different K loci.
BEST_EXCLUDED = {
    p.stem: p
    for d in ("06_asm_fixing/01_autocycler/assemblies",
              "06_asm_fixing/01_autocycler/failed",
              "06_asm_fixing/02_decontam/assemblies")
    for p in Path(d).glob("*.fasta")
}

iq = internal_qc.set_index("sample")
rows = []
for lib, iso in zip(excluded.library, excluded.isolate):
    path = BEST_EXCLUDED.get(lib) or BEST_EXCLUDED.get(iso)
    if path is not None:
        lengths = [len(seq) for _, seq in parse_fasta(path)]
        chrom = [n for n in lengths if n >= CHROMOSOME_MIN_LEN]
        rows.append((len(lengths), sum(lengths),
                     chrom[0] if len(chrom) == 1 else "", f"remediation ({path.parent.name})"))
    else:
        rows.append((iq.asm_num_seqs[lib], iq.asm_sum_len[lib],
                     iq.asm_max_len[lib] if iq.asm_n_chromosomes[lib] == 1 else "",
                     "first-pass Flye"))

excluded[["ont_contigs", "ont_total_len", "ont_chromosome_len"]] = [r[:3] for r in rows]
excluded["released"] = "no"
for (lib, iso), r in zip(zip(excluded.library, excluded.isolate), rows):
    if r[3] != "first-pass Flye":
        print(f"  {iso:<10} ONT stats from {r[3]}: {r[0]} contigs, {r[1]:,} bp "
              f"(first-pass Flye was {iq.asm_num_seqs[lib]} contigs)")

s5 = pd.concat([s5_rel, excluded], ignore_index=True)

# Figures 3 and 4 take their per-library quantities from this table and from nowhere
# else, so the metrics behind the internal verdict travel with it.
s5 = s5.merge(verification_metrics, on="library", how="left", validate="1:1")
unmeasured = sorted(s5.loc[s5.mapped_pct.isna(), "library"])
if unmeasured:
    raise ValueError(f"libraries in S5 with no verification metrics: {unmeasured}")

S5_COLS = {
    "isolate": "Biological sample.",
    "library": "One sequenced preparation.",
    "colony_pick": "`.1`/`.2` where the isolate yielded two colony morphologies; empty otherwise.",
    "resequenced": "`yes` for the 131_RM batch.",
    "internal_qc_decision": "Verdict from the four-predicate internal gate. 288 libraries were decided automatically; the 28 that were not are in Table S3.",
    "mapped_pct": "Percentage of this library's Illumina reads mapping to its Nanopore-only assembly. STALE FOR THE FOUR DECONTAMINATED LIBRARIES: Illumina was mapped against the pre-decontamination assemblies. That is the measurement internal QC judged, and it is correct for this row's verdict, but it is not a property of the released genome and must not be quoted as one.",
    "properly_paired_pct": "Percentage of this library's Illumina reads mapping as a proper pair, same alignment.",
    "klebsiella_genus_read_fraction": "Share of the library's Illumina reads Kraken2 assigns to genus Klebsiella. A FRACTION, 0-1, not a percentage like the columns above it, and the denominator is all reads, unclassified included. The screen flags a library below 0.50.",
    "top_non_klebsiella_genus": "The highest-scoring genus outside the Klebsiella/Raoultella group, from the same genus walk and the same exclusion set as bin/scripts/kraken2_parser.py, so this column and kraken2_status read the report the same way. Every library has one; for most it is a trace.",
    "top_non_klebsiella_genus_fraction": "Share of reads assigned to that genus. A FRACTION, 0-1. The contamination flag fires at 0.05, which 9 libraries reach.",
    "kraken2_status": "PASS or FAIL from the Kraken2 contamination screen, at the thresholds pinned in bin/08_run_kraken2.sh and bin/scripts/kraken2_parser.py.",
    "ont_species": "Species called by Kleborate on the first-pass Nanopore-only assembly - the assembly internal QC judged. NOT interchangeable with genome_typing.tsv's column of the same name, which reports the call on the assembly the released genome was finally built from: they differ for INF186 and INF216, the libraries reassembled from decontaminated reads. INF092.2 reads NO_KLEBORATE_CALL: that library is 95% Enterobacter and Kleborate returned nothing for it. That is a value, not a gap, and a figure has to render it as one.",
    "ont_ST": "Sequence type on that same first-pass Nanopore-only assembly.",
    "ont_K_locus": "K locus on that same first-pass Nanopore-only assembly.",
    "illumina_species": "Species called by Kleborate on the Illumina-only assembly. Agrees with genome_typing.tsv for all 294 released libraries.",
    "illumina_ST": "Sequence type on the Illumina-only assembly.",
    "illumina_K_locus": "K locus on the Illumina-only assembly.",
    "external_qc_decision": "Verdict from comparison against the collection's historical typing. Empty where the library was dropped before this stage.",
    "external_mismatch_reason": "Why the external verdict was not a plain PASS.",
    "final_qc_decision": "Verdict after hybrid assembly and polishing. Empty for excluded libraries.",
    "read_prep": "`standard`, or `kraken2_decontam` where the ONT reads had a second organism removed. Empty for excluded libraries.",
    "assembly_route": "Which of the four routes produced the genome. Empty for excluded libraries.",
    "hybrid_contigs": "Contigs in the released hybrid assembly.",
    "hybrid_total_len": "Total length of the released hybrid assembly.",
    "hybrid_chromosome_len": "Chromosome length in the released hybrid assembly.",
    "ont_contigs": "Contigs in the best Nanopore-only assembly the library reached. For released libraries that is the assembly the hybrid was polished from. For excluded libraries it is the remediation assembly where one was attempted (INF096 and INF294 went through Autocycler; INF262 and KSB2_5D through decontamination) and the first-pass Flye assembly otherwise - so a reader can see that INF096 and INF294 assembled acceptably and were dropped on typing, not on assembly quality.",
    "ont_total_len": "Total length of that Nanopore-only assembly.",
    "ont_chromosome_len": "Chromosome length in that Nanopore-only assembly. Empty for an excluded library whose assembly had no single contig >= 4 Mb.",
    "d_contigs": "hybrid_contigs minus ont_contigs: the change polishing made.",
    "d_total_len": "hybrid_total_len minus ont_total_len.",
    "d_chromosome_len": "hybrid_chromosome_len minus ont_chromosome_len. The chromosome is considered preserved within 50 kb (MAX_CHROMOSOME_DELTA in 09_final_qc/final_qc.ipynb).",
    "released": "`yes` for the 294 libraries whose assembly is in the released set.",
    "exclusion_reason": "Matches sample_manifest.tsv. Empty where released.",
}
S5 = write_table(s5, "quality_control_and_routing", S5_COLS,
                 [SRC["final_qc"], SRC["roster"], SRC["internal_qc"],
                  SRC["verification"], SRC["kraken2_reports"],
                  TABLES / "external_qc_summary.tsv"])

  INF096     ONT stats from remediation (failed): 2 contigs, 5,469,159 bp (first-pass Flye was 30 contigs)
  INF294     ONT stats from remediation (failed): 3 contigs, 5,690,842 bp (first-pass Flye was 23 contigs)
  KSB2_5D    ONT stats from remediation (assemblies): 4 contigs, 5,535,570 bp (first-pass Flye was 103 contigs)

Table_S5_qc_and_routing.tsv — 316 rows × 33 columns from ../pipeline_tables/final_qc_summary.tsv, ../pipeline_tables/KASPAH_QC_outcome_v2.tsv, ../pipeline_tables/internal_qc_summary.tsv, ../pipeline_tables/verification_summary.tsv, 04_verification/taxonomy/output, ../pipeline_tables/external_qc_summary.tsv


In [15]:
check("S5", "316 rows", len(S5) == 316, len(S5))
check("S5", "released == yes for 294", (S5.released == "yes").sum() == 294,
      (S5.released == "yes").sum())
exc = S5[S5.released == "no"]
check("S5", "22 excluded split 18 duplicate_colony_pick + 4 isolate losses",
      (exc.exclusion_reason == "duplicate_colony_pick").sum() == 18 and len(exc) == 22,
      exc.exclusion_reason.value_counts().to_dict())
check("S5", "the four isolate losses are INF096, INF294, INF262, KSB2_5D",
      set(exc.loc[exc.exclusion_reason != "duplicate_colony_pick", "isolate"])
      == {"INF096", "INF294", "INF262", "KSB2_5D"},
      sorted(exc.loc[exc.exclusion_reason != "duplicate_colony_pick", "isolate"]))
rel5 = S5[S5.released == "yes"]
check("S5", "assembly_route counts 283 / 6 / 4 / 1",
      rel5.assembly_route.value_counts().to_dict() == {
          "clinopore": 283, "flye (03_assembly) > repolish": 6,
          "kraken2 decontam > clinopore": 4, "autocycler > repolish": 1},
      rel5.assembly_route.value_counts().to_dict())
check("S5", "external_qc_decision over released rows is 279 PASS + 15 PASS_REFERENCE_MISMATCH",
      rel5.external_qc_decision.value_counts().to_dict() == {
          "PASS": 279, "PASS_REFERENCE_MISMATCH": 15},
      rel5.external_qc_decision.value_counts().to_dict())
check("S5", "final_qc_decision == PASS for all 294",
      set(rel5.final_qc_decision) == {"PASS"}, rel5.final_qc_decision.value_counts().to_dict())
check("S5", "exclusion_reason matches S2 exactly",
      S5.set_index("library").exclusion_reason.sort_index().equals(
          S2.set_index("library").exclusion_reason.sort_index()))
check("S5", "excluded rows carry no hybrid or delta values",
      (exc[["final_qc_decision", "read_prep", "assembly_route", "hybrid_contigs",
            "d_total_len"]] == "").all().all())
check("S5", "excluded rows do carry ONT assembly statistics",
      (exc[["ont_contigs", "ont_total_len"]] != "").all().all(),
      sorted(exc.loc[exc.ont_contigs == "", "library"]))
check("S5", "INF096 and INF294 assembled acceptably and were excluded on typing",
      all(int(exc.loc[exc.isolate == iso, "ont_contigs"].iloc[0]) <= 3
          and exc.loc[exc.isolate == iso, "exclusion_reason"].iloc[0]
          == "platform_k_locus_discordance" for iso in ("INF096", "INF294")))
for iso in ("INF096", "INF294", "INF262", "KSB2_5D"):
    row = exc[exc.isolate == iso].iloc[0]
    print(f"  {iso:<9} {row.ont_contigs:>2} contigs, {int(row.ont_total_len):>9,} bp, "
          f"chromosome {row.ont_chromosome_len or 'none >= 4 Mb':>9} — {row.exclusion_reason}")
check("S5", "18 released rows carry a colony_pick",
      (rel5.colony_pick != "").sum() == 18, (rel5.colony_pick != "").sum())

METRIC_COLS = ["mapped_pct", "properly_paired_pct", "klebsiella_genus_read_fraction",
               "top_non_klebsiella_genus", "top_non_klebsiella_genus_fraction",
               "kraken2_status", "ont_species", "ont_ST", "ont_K_locus",
               "illumina_species", "illumina_ST", "illumina_K_locus"]
empty = {c: sorted(S5.loc[S5[c].astype(str).str.strip() == "", "library"])
         for c in METRIC_COLS}
empty = {c: libs for c, libs in empty.items() if libs}
check("S5", "no blank cell in any of the 12 metric columns, over all 316 rows",
      not empty, empty)

# The metrics have to reproduce the gate they fed. Each count below is a predicate in Table
# S4: if one of these moves, an adjudication moved with it.
check("S5", "mapped_pct: 12 libraries below the 95% gate, range 11.69-99.99",
      (s5.mapped_pct < 95).sum() == 12
      and (round(s5.mapped_pct.min(), 2), round(s5.mapped_pct.max(), 2)) == (11.69, 99.99),
      f"{(s5.mapped_pct < 95).sum()} below 95, "
      f"range {s5.mapped_pct.min():.2f}-{s5.mapped_pct.max():.2f}")
check("S5", "kraken2_status is 300 PASS + 16 FAIL",
      S5.kraken2_status.value_counts().to_dict() == {"PASS": 300, "FAIL": 16},
      S5.kraken2_status.value_counts().to_dict())
check("S5", "6 libraries fall below the 0.50 Klebsiella-fraction floor",
      (s5.klebsiella_genus_read_fraction < 0.50).sum() == 6,
      sorted(s5.loc[s5.klebsiella_genus_read_fraction < 0.50, "library"]))
check("S5", "12 libraries type to different K loci on the two platforms, matching the S4 predicate",
      (s5.ont_K_locus != s5.illumina_K_locus).sum() == 12,
      (s5.ont_K_locus != s5.illumina_K_locus).sum())
check("S5", "7 libraries type to different species on the two platforms",
      (s5.ont_species != s5.illumina_species).sum() == 7,
      sorted(s5.loc[s5.ont_species != s5.illumina_species, "library"]))
# NO_KLEBORATE_CALL is a value, not a blank, so the no-blank-cell check above cannot see it.
# It counts as a disagreement in both figures above. One library carries it; if a second
# ever does, this is where it surfaces rather than in a figure.
no_call = sorted(s5.loc[s5.ont_species == "NO_KLEBORATE_CALL", "library"])
check("S5", "exactly one library, INF092.2, has no ONT Kleborate call",
      no_call == ["INF092.2"], no_call)

[  ok  ] S5     316 rows  — 316
[  ok  ] S5     released == yes for 294  — 294
[  ok  ] S5     22 excluded split 18 duplicate_colony_pick + 4 isolate losses  — {'duplicate_colony_pick': 18, 'platform_k_locus_discordance': 2, 'library_not_labelled_isolate': 1, 'platform_species_discordance': 1}
[  ok  ] S5     the four isolate losses are INF096, INF294, INF262, KSB2_5D  — ['INF096', 'INF262', 'INF294', 'KSB2_5D']
[  ok  ] S5     assembly_route counts 283 / 6 / 4 / 1  — {'clinopore': 283, 'flye (03_assembly) > repolish': 6, 'kraken2 decontam > clinopore': 4, 'autocycler > repolish': 1}
[  ok  ] S5     external_qc_decision over released rows is 279 PASS + 15 PASS_REFERENCE_MISMATCH  — {'PASS': 279, 'PASS_REFERENCE_MISMATCH': 15}
[  ok  ] S5     final_qc_decision == PASS for all 294  — {'PASS': 294}
[  ok  ] S5     exclusion_reason matches S2 exactly
[  ok  ] S5     excluded rows carry no hybrid or delta values
[  ok  ] S5     excluded rows do carry ONT assembly statistics
[  ok  ] S5     

True

## S6 — Typing from all four sources

A straight column projection of `final_qc_summary.tsv`: no filtering, no recomputation, no
normalisation. The typing is the **hybrid assemblies' own Kleborate calls**; `reference_*`
is the historical short-read typing, present for comparison only.

ST values keep Kleborate's `-nLV` nearest-neighbour suffixes. Platform concordance in the
manuscript is assessed *after* stripping them, so anyone reproducing the 294/294 figure from
this table has to normalise — which is why the column definitions say so.

In [16]:
S6_SRC_COLS = [
    "isolate", "library", "hybrid_species", "hybrid_ST", "hybrid_K_locus",
    "hybrid_K_locus_confidence", "hybrid_O_locus", "ont_species", "ont_ST", "ont_K_locus",
    "illumina_species", "illumina_ST", "illumina_K_locus", "reference_species",
    "reference_ST", "reference_K_locus", "matches_reference",
    "mlst_distance_ont_to_hybrid", "mlst_distance_illumina_to_hybrid",
    "mlst_distance_reference_to_hybrid",
]
s6 = final_qc[S6_SRC_COLS].copy()
s6["matches_reference"] = yn(s6.matches_reference.astype(bool))

S6_COLS = {
    "isolate": "Biological sample.",
    "library": "The sequenced preparation the released assembly was built from.",
    "hybrid_species": "Species or subspecies called by Kleborate on the released hybrid assembly.",
    "hybrid_ST": "Sequence type called on the hybrid assembly. `-nLV` marks a nearest-neighbour call n loci from a named ST.",
    "hybrid_K_locus": "K locus called on the hybrid assembly.",
    "hybrid_K_locus_confidence": "Kleborate's confidence in the K-locus call. `Untypeable` calls retain a locus name and are counted as distinct values in the diversity figures quoted in Technical Validation.",
    "hybrid_O_locus": "O locus called on the hybrid assembly.",
    "ont_species": "Species called on the Nanopore-only assembly.",
    "ont_ST": "Sequence type called on the Nanopore-only assembly.",
    "ont_K_locus": "K locus called on the Nanopore-only assembly.",
    "illumina_species": "Species called on the Illumina-only assembly.",
    "illumina_ST": "Sequence type called on the Illumina-only assembly.",
    "illumina_K_locus": "K locus called on the Illumina-only assembly.",
    "reference_species": "Species in the collection's historical short-read typing.",
    "reference_ST": "Sequence type in the historical typing.",
    "reference_K_locus": "K locus in the historical typing.",
    "matches_reference": "`no` where the historical typing disagrees with the assembly. In every such case the evidence favours the assembly; see 07_external_qc/README.md.",
    "mlst_distance_ont_to_hybrid": "Number of the seven MLST loci whose allele differs between the Nanopore-only and hybrid assemblies. Inexact markers stripped; a locus missing on either side counts as a difference.",
    "mlst_distance_illumina_to_hybrid": "The same, Illumina-only against hybrid.",
    "mlst_distance_reference_to_hybrid": "The same, historical typing against hybrid. Concordance threshold is <= 2; the observed distribution has nothing at 2.",
}
S6 = write_table(s6, "genome_typing", S6_COLS, [SRC["final_qc"]])


Table_S6_typing.tsv — 294 rows × 20 columns from ../pipeline_tables/final_qc_summary.tsv


In [17]:
check("S6", "294 rows", len(S6) == 294, len(S6))
strip_lv = lambda s: s.str.replace(r"-\d+LV$", "", regex=True)
agree = (strip_lv(s6.ont_ST) == strip_lv(s6.illumina_ST)) & \
        (s6.ont_species == s6.illumina_species) & (s6.ont_K_locus == s6.illumina_K_locus)
check("S6", "ONT and Illumina agree on species, ST and K locus for all 294 after -nLV normalisation",
      agree.all(), f"{(~agree).sum()} disagree: {list(s6.loc[~agree, 'isolate'])}")
check("S6", "matches_reference == no for 15", (S6.matches_reference == "no").sum() == 15,
      (S6.matches_reference == "no").sum())
mism = s6[s6.matches_reference == "no"]
check("S6", "of the 15 mismatches, 3 differ only at the K locus",
      (mism.mlst_distance_reference_to_hybrid == 0).sum() == 3,
      sorted(mism.loc[mism.mlst_distance_reference_to_hybrid == 0, "isolate"]))
check("S6", "no reference-to-hybrid MLST distance equals 2",
      (s6.mlst_distance_reference_to_hybrid == 2).sum() == 0,
      s6.mlst_distance_reference_to_hybrid.value_counts().sort_index().to_dict())

# The diversity figures the Technical Validation paragraph quotes are counted straight off
# this table now that the per-locus count table is gone. The manuscript footnotes the calls
# that are not straightforward — a nearest-neighbour ST and three K loci below Kleborate's
# confidence criteria — and all four are counted as distinct values, which is why the
# footnote is needed.
check("S6", "212 distinct STs / 96 distinct K loci / 10 distinct O loci",
      (s6.hybrid_ST.nunique(), s6.hybrid_K_locus.nunique(), s6.hybrid_O_locus.nunique())
      == (212, 96, 10),
      (s6.hybrid_ST.nunique(), s6.hybrid_K_locus.nunique(), s6.hybrid_O_locus.nunique()))
st_counts = s6.hybrid_ST.value_counts()
check("S6", "177 STs seen once, ST323 the most common at 8",
      (st_counts == 1).sum() == 177 and st_counts.max() == 8
      and st_counts.idxmax() == "ST323",
      f"{(st_counts == 1).sum()} singletons, most common {st_counts.idxmax()} "
      f"= {st_counts.max()}")
k_counts = s6.hybrid_K_locus.value_counts()
check("S6", "40 K loci seen once, KL30 the most common at 15",
      (k_counts == 1).sum() == 40 and k_counts["KL30"] == 15,
      f"{(k_counts == 1).sum()} singletons, KL30 = {k_counts['KL30']}")

low_conf_k = s6.loc[s6.hybrid_K_locus_confidence.isin(["Untypeable", "Low", "None"]),
                    ["isolate", "hybrid_K_locus", "hybrid_K_locus_confidence"]]
nn_st = s6.loc[s6.hybrid_ST.str.contains(r"-\d+LV$", regex=True), ["isolate", "hybrid_ST"]]
check("S6", "3 low-confidence K-locus calls and 1 nearest-neighbour ST, all footnoted",
      len(low_conf_k) == 3 and len(nn_st) == 1,
      f"{len(low_conf_k)} low-confidence K loci, {len(nn_st)} nearest-neighbour STs")
print()
print(low_conf_k.to_string(index=False))
print(nn_st.to_string(index=False))

[  ok  ] S6     294 rows  — 294
[  ok  ] S6     ONT and Illumina agree on species, ST and K locus for all 294 after -nLV normalisation  — 0 disagree: []
[  ok  ] S6     matches_reference == no for 15  — 15
[  ok  ] S6     of the 15 mismatches, 3 differ only at the K locus  — ['INF033', 'INF267', 'INF311']
[  ok  ] S6     no reference-to-hybrid MLST distance equals 2  — {0: 280, 1: 3, 3: 1, 4: 2, 5: 3, 6: 1, 7: 4}
[  ok  ] S6     212 distinct STs / 96 distinct K loci / 10 distinct O loci  — (212, 96, 10)
[  ok  ] S6     177 STs seen once, ST323 the most common at 8  — 177 singletons, most common ST323 = 8
[  ok  ] S6     40 K loci seen once, KL30 the most common at 15  — 40 singletons, KL30 = 15
[  ok  ] S6     3 low-confidence K-locus calls and 1 nearest-neighbour ST, all footnoted  — 3 low-confidence K loci, 1 nearest-neighbour STs

isolate hybrid_K_locus hybrid_K_locus_confidence
 INF181           KL14                Untypeable
 INF311          KL144                Untypeable
KSB1_3C

## S8 — Per-assembly statistics

Built before read_statistics.tsv because read depth is bases over assembly length.

The 294 released FASTAs carry `depth=`, `circular=` and `length=` in their headers, carried
through from the source long-read assembly — polishing does not re-derive topology, so this
is the only place circularity exists after the hybrid stage. GC is computed from the
sequence.

The per-contig frame this is built from is **no longer published** — the per-contig table
that used to ship as S5b has been dropped — but it is still computed, because every
per-assembly figure here and the contig total quoted in Data Records come from it.

**INF205 is the exception and must stay one.** It is the Autocycler consensus, and its
headers carry no `circular=` flag at all. It is reported `unknown`, not `no`. Coding it as
`no` moves the headline completeness figure.

**Chromosome is a contig ≥ 4 Mb**, applied identically everywhere. In this collection
chromosomes run 5.03–5.66 Mb and the largest extrachromosomal contig is far smaller, so
nothing falls near the cut.

### CheckM2 completeness and contamination

`checkm2_completeness` and `checkm2_contamination` come from a single CheckM2 run over the
294 released FASTAs (`09_final_qc/completeness_final.sh`). They are joined, not recomputed:
the report's `Genome_Size` and `Total_Contigs` are checked against `total_length_bp` and
`contigs` below, so a stale report cannot pass unnoticed.

Completeness is 99.99–100.00% across the collection and carries no information; it is
published because a completeness column absent from an assembly table invites the reader to
assume it was never measured.

**Contamination is scored over every contig given to CheckM2, so it is a property of the
released assembly, not of the chromosome.** Median 0.20%, and 280 of 294 sit below 1%.
Three exceed 5% — INF035 (5.32), INF074 (5.45) and KSB2_6A (10.83). Re-running CheckM2 on
the chromosome alone (`09_final_qc/completeness_chromosome.sh`) drops all three to 0.06,
0.09 and 0.12: the duplicated single-copy markers are on extrachromosomal contigs, and all
three chromosomes are clean. KSB2_6A is the one that also loses completeness, 100.00 →
91.73, so some of its 1.34 Mb of accessory sequence is chromosomal material the assembly
did not place. The published columns keep the whole-assembly figures, because that is what
the deposited FASTA contains.

In [18]:
contig_rows = []
for fasta in sorted(Path(SRC["assemblies"]).glob("*.fasta")):
    isolate = fasta.stem
    for header, seq in parse_fasta(fasta):
        name = header.split()[0]
        attrs = dict(
            kv.split("=", 1) for kv in header.split()[1:] if "=" in kv
        )
        circ = attrs.get("circular")
        contig_rows.append({
            "isolate": isolate,
            "contig": name,
            "length_bp": len(seq),
            "gc_percent": round(gc_percent(seq), 2),
            "circular": {"Y": "yes", "N": "no"}.get(circ, "unknown"),
            "mean_depth_x": float(attrs["depth"]) if "depth" in attrs else None,
            "class": "chromosome" if len(seq) >= CHROMOSOME_MIN_LEN else "extrachromosomal",
        })

contig_stats = pd.DataFrame(contig_rows)
print(f"{contig_stats.isolate.nunique()} assemblies · {len(contig_stats)} contigs")
print(f"contigs with no circularity flag: "
      f"{sorted(contig_stats[contig_stats.circular == 'unknown'].isolate.unique())}")


294 assemblies · 837 contigs
contigs with no circularity flag: ['INF205']


In [19]:
rows = []
for isolate, grp in contig_stats.groupby("isolate"):
    chrom = grp[grp["class"] == "chromosome"]
    extra = grp[grp["class"] == "extrachromosomal"]
    total = grp.length_bp.sum()
    rows.append({
        "isolate": isolate,
        "contigs": len(grp),
        "total_length_bp": int(total),
        "n50_bp": int(n50(grp.length_bp)),
        "gc_percent": round(
            float((grp.gc_percent * grp.length_bp).sum() / total), 2),
        "chromosome_length_bp": int(chrom.length_bp.iloc[0]) if len(chrom) == 1 else None,
        "chromosome_gc_percent": float(chrom.gc_percent.iloc[0]) if len(chrom) == 1 else None,
        "chromosome_circular": chrom.circular.iloc[0] if len(chrom) == 1 else "unknown",
        "chromosome_mean_depth_x": chrom.mean_depth_x.iloc[0] if len(chrom) == 1 else None,
        "extrachromosomal_contigs": len(extra),
        "all_contigs_circular": collapse_circular(grp.circular),
    })

s8 = pd.DataFrame(rows)
s8 = s8.merge(
    final_qc[["isolate", "library", "assembly_route"]], on="isolate",
    how="left", validate="1:1")

# CheckM2 over the released FASTAs. `Name` is the FASTA basename, which is the isolate.
checkm2 = (pd.read_csv(SRC["checkm2"], sep="\t")
           .rename(columns={"Name": "isolate",
                            "Completeness": "checkm2_completeness",
                            "Contamination": "checkm2_contamination",
                            "Genome_Size": "checkm2_genome_size",
                            "Total_Contigs": "checkm2_contigs"}))
s8 = s8.merge(
    checkm2[["isolate", "checkm2_completeness", "checkm2_contamination",
             "checkm2_genome_size", "checkm2_contigs"]],
    on="isolate", how="left", validate="1:1")

S8_COLS = {
    "isolate": "Biological sample; names the released FASTA.",
    "library": "The one sequenced preparation this genome was built from.",
    "contigs": "Contigs in the released assembly.",
    "total_length_bp": "Sum of contig lengths.",
    "n50_bp": "Contig N50.",
    "gc_percent": "Length-weighted GC content of the whole assembly.",
    "checkm2_completeness": "CheckM2 completeness of the released assembly, percent.",
    "checkm2_contamination": "CheckM2 contamination of the released assembly, percent. Scored over every contig, so duplicated single-copy markers on extrachromosomal contigs count against the genome; the three isolates above 5% score 0.06-0.12 on the chromosome alone.",
    "chromosome_length_bp": "Length of the single contig >= 4 Mb.",
    "chromosome_gc_percent": "GC content of the chromosome.",
    "chromosome_circular": "`yes`/`no` from the chromosome's `circular=` flag; `unknown` for INF205, whose Autocycler consensus headers carry no flag. This column is the source of the completeness figure the manuscript quotes.",
    "chromosome_mean_depth_x": "Mean read depth over the chromosome. Empty where the headers carry none.",
    "extrachromosomal_contigs": "Contigs below 4 Mb. Zero means the isolate had nothing extrachromosomal to assemble, not a better assembly.",
    "all_contigs_circular": "`yes` only where every contig is flagged circular; `unknown` where any contig's topology is unknown.",
    "assembly_route": "Which of the four routes produced this genome; joined from quality_control_and_routing.tsv.",
}
S8 = write_table(s8, "assembly_statistics", S8_COLS,
                 [SRC["assemblies"], SRC["final_qc"], SRC["checkm2"]])


Table_S8_per_assembly_statistics.tsv — 294 rows × 15 columns from 10_final_assemblies/assemblies, ../pipeline_tables/final_qc_summary.tsv, ../pipeline_tables/checkm2_quality_report.tsv


In [20]:
check("S8", "294 rows", len(S8) == 294, len(S8))
per_iso_chrom = contig_stats[contig_stats["class"] == "chromosome"].groupby("isolate").size()
check("S8", "exactly one chromosome per isolate",
      len(per_iso_chrom) == 294 and (per_iso_chrom == 1).all(),
      f"{len(per_iso_chrom)} isolates, max {per_iso_chrom.max()}")
chrom_len = s8.chromosome_length_bp
check("S8", "chromosome_length_bp range 5,031,183–5,660,135",
      (chrom_len.min(), chrom_len.max()) == (5031183, 5660135),
      f"{chrom_len.min():,}–{chrom_len.max():,}")
# The manuscript quotes 5,318,719; the value is 5,318,718.47, so its integer is 5,318,718.
# A 1 bp difference on a 5.3 Mb mean is a rounding convention, not a discrepancy, and the
# check allows it rather than failing on every rebuild.
check("S8", "chromosome_length_bp mean 5,318,719 (+/-1 bp) / median 5,307,218",
      abs(chrom_len.mean() - 5318719) <= 1 and chrom_len.median() == 5307218,
      f"mean {chrom_len.mean():,.2f} median {chrom_len.median():,.0f}")
check("S8", "contigs mean 2.8 / median 2 / range 1–16",
      round(s8.contigs.mean(), 1) == 2.8 and s8.contigs.median() == 2
      and (s8.contigs.min(), s8.contigs.max()) == (1, 16),
      f"mean {s8.contigs.mean():.2f} median {s8.contigs.median()} "
      f"range {s8.contigs.min()}–{s8.contigs.max()}")
tot = s8.total_length_bp
# The median is exactly 5,519,564.5 — an even number of genomes — so 5,519,565 is a half-up
# rounding of a real tie rather than a wrong number.
check("S8", "total_length_bp mean 5,529,507 / median 5,519,565 (+/-1 bp) / range 5,095,397–6,488,464",
      round(tot.mean()) == 5529507 and abs(tot.median() - 5519565) <= 1
      and (tot.min(), tot.max()) == (5095397, 6488464),
      f"mean {tot.mean():,.0f} median {tot.median():,.1f} range {tot.min():,}–{tot.max():,}")
check("S8", "chromosome GC mean 57.5%, range 57.1–58.3%",
      round(s8.chromosome_gc_percent.mean(), 1) == 57.5
      and round(s8.chromosome_gc_percent.min(), 1) >= 57.1
      and round(s8.chromosome_gc_percent.max(), 1) <= 58.3,
      f"mean {s8.chromosome_gc_percent.mean():.2f} "
      f"range {s8.chromosome_gc_percent.min():.2f}–{s8.chromosome_gc_percent.max():.2f}")
check("S8", "chromosome_circular == yes for 276",
      (s8.chromosome_circular == "yes").sum() == 276,
      s8.chromosome_circular.value_counts().to_dict())
check("S8", "chromosome_circular == unknown for 1 (INF205)",
      list(s8.loc[s8.chromosome_circular == "unknown", "isolate"]) == ["INF205"],
      list(s8.loc[s8.chromosome_circular == "unknown", "isolate"]))
check("S8", "all_contigs_circular == yes for 224",
      (s8.all_contigs_circular == "yes").sum() == 224,
      s8.all_contigs_circular.value_counts().to_dict())
check("S8", "contigs == 1 for 60", (s8.contigs == 1).sum() == 60, (s8.contigs == 1).sum())
check("S8", "extrachromosomal_contigs >= 1 for 234, mean 1.8, max 15",
      (s8.extrachromosomal_contigs >= 1).sum() == 234
      and round(s8.extrachromosomal_contigs.mean(), 1) == 1.8
      and s8.extrachromosomal_contigs.max() == 15,
      f"{(s8.extrachromosomal_contigs >= 1).sum()} with >=1, mean "
      f"{s8.extrachromosomal_contigs.mean():.2f}, max {s8.extrachromosomal_contigs.max()}")
counts = contig_stats.groupby("isolate").size()
check("S8", "contigs equals the per-contig frame's row count per isolate",
      s8.set_index("isolate").contigs.eq(counts).all())
hyb = final_qc.set_index("isolate")
check("S8/S5", "total_length_bp equals hybrid_total_len for every isolate",
      s8.set_index("isolate").total_length_bp.eq(hyb.hybrid_total_len).all(),
      f"{(~s8.set_index('isolate').total_length_bp.eq(hyb.hybrid_total_len)).sum()} differ")
check("S8/S5", "chromosome_length_bp equals hybrid_chromosome_len for every isolate",
      s8.set_index("isolate").chromosome_length_bp.eq(hyb.hybrid_chromosome_len).all(),
      f"{(~s8.set_index('isolate').chromosome_length_bp.eq(hyb.hybrid_chromosome_len)).sum()} differ")

# CheckM2 is joined from a separate run, so the join is checked before the values are.
# Genome_Size and Total_Contigs are CheckM2's own measurements of the FASTAs it was given:
# if they match this table, the report describes these assemblies and not an earlier build.
check("S8", "every isolate has a CheckM2 result",
      s8[["checkm2_completeness", "checkm2_contamination"]].notna().all().all(),
      f"{s8.checkm2_completeness.isna().sum()} missing")
check("S8", "CheckM2 scored the released assemblies",
      s8.checkm2_genome_size.eq(s8.total_length_bp).all()
      and s8.checkm2_contigs.eq(s8.contigs).all(),
      f"{(~s8.checkm2_genome_size.eq(s8.total_length_bp)).sum()} length "
      f"/ {(~s8.checkm2_contigs.eq(s8.contigs)).sum()} contig differences")
comp = s8.checkm2_completeness
check("S8", "checkm2_completeness 99.99-100.00 for all 294",
      (comp.min(), comp.max()) == (99.99, 100.0),
      f"{comp.min():.2f}-{comp.max():.2f}, {(comp == 100.0).sum()} at 100.00")
cont = s8.checkm2_contamination
check("S8", "checkm2_contamination median 0.20, range 0.01-10.83",
      cont.median() == 0.20 and (cont.min(), cont.max()) == (0.01, 10.83),
      f"median {cont.median():.2f} range {cont.min():.2f}-{cont.max():.2f}")
check("S8", "checkm2_contamination < 1% for 280 of 294",
      (cont < 1).sum() == 280, (cont < 1).sum())
check("S8", "checkm2_contamination > 5% for INF035, INF074, KSB2_6A",
      sorted(s8.loc[cont > 5, "isolate"]) == ["INF035", "INF074", "KSB2_6A"],
      sorted(s8.loc[cont > 5, "isolate"]))


[  ok  ] S8     294 rows  — 294
[  ok  ] S8     exactly one chromosome per isolate  — 294 isolates, max 1
[  ok  ] S8     chromosome_length_bp range 5,031,183–5,660,135  — 5,031,183–5,660,135
[  ok  ] S8     chromosome_length_bp mean 5,318,719 (+/-1 bp) / median 5,307,218  — mean 5,318,718.47 median 5,307,218
[  ok  ] S8     contigs mean 2.8 / median 2 / range 1–16  — mean 2.85 median 2.0 range 1–16
[  ok  ] S8     total_length_bp mean 5,529,507 / median 5,519,565 (+/-1 bp) / range 5,095,397–6,488,464  — mean 5,529,507 median 5,519,564.5 range 5,095,397–6,488,464
[  ok  ] S8     chromosome GC mean 57.5%, range 57.1–58.3%  — mean 57.50 range 57.12–58.28
[  ok  ] S8     chromosome_circular == yes for 276  — {'yes': 276, 'no': 17, 'unknown': 1}
[  ok  ] S8     chromosome_circular == unknown for 1 (INF205)  — ['INF205']
[  ok  ] S8     all_contigs_circular == yes for 224  — {'yes': 224, 'no': 69, 'unknown': 1}
[  ok  ] S8     contigs == 1 for 60  — 60
[  ok  ] S8     extrachromosomal_conti

True

## S7 — Per-isolate read statistics

The statistics must describe **the reads each assembly was actually built from**: post-Filtlong,
and for the four decontaminated libraries the Kraken2-filtered set rather than the raw library.
`ont_read_set` records which. Getting this wrong overstates yield for exactly the four
libraries where yield was the problem.

Depth is bases over the isolate's own final assembly length from assembly_statistics.tsv, not over a nominal
5.5 Mb.

In [21]:
ont_stats = pd.concat([
    pd.read_csv(SRC["ont_stats_v1"], sep="\t"),
    pd.read_csv(SRC["ont_stats_v2"], sep="\t"),
], ignore_index=True)


def ont_library(path):
    """`INF001.1/qc/…`, `02_reads/INF034_v2/qc/…` and the decontam read sets."""
    parts = Path(path).parts
    if "06_asm_fixing" in parts:
        return Path(path).name.split(".ont.decontam")[0]
    return parts[1] if parts[0] == "02_reads" else parts[0]


ont_stats["library"] = ont_stats.file.map(ont_library)
ont_stats["ont_read_set"] = ont_stats.file.map(
    lambda p: "kraken2_decontam_filtered" if "06_asm_fixing" in p else "filtered")
# where a library has both a raw and a decontaminated set, the assembly used the latter
ont_stats = (ont_stats.sort_values("ont_read_set", ascending=False)
             .drop_duplicates("library", keep="first"))

released = final_qc[["isolate", "library"]].copy()
missing_reads = set(released.library) - set(ont_stats.library)
check("S7", "every released library has ONT read statistics",
      not missing_reads, sorted(missing_reads))

s7 = released.merge(ont_stats, on="library", how="left", validate="1:1")

def fastp_report(library):
    """The v1 and v2 read libraries name their fastp reports differently.

    v1 writes `<library>_fastp.json`; the 19 resequenced libraries write `fastp_report.json`.
    Missing the second spelling drops those 19 from every Illumina statistic and pulls the
    collection means down by ~6%, silently, because they simply become null.
    """
    for name in (f"{library}_fastp.json", "fastp_report.json"):
        path = Path(f"02_reads/{library}/qc/{name}")
        if path.exists():
            return path
    return None


fastp_rows = []
for lib in s7.library:
    path = fastp_report(lib)
    if path is None:
        fastp_rows.append({"library": lib})
        continue
    rep = json.loads(path.read_text())
    after, before = rep["summary"]["after_filtering"], rep["summary"]["before_filtering"]
    fastp_rows.append({
        "library": lib,
        "illumina_read_pairs": after["total_reads"] // 2,
        "illumina_bases": after["total_bases"],
        "illumina_mean_read_length_bp": round(
            (after["read1_mean_length"] + after["read2_mean_length"]) / 2),
        "illumina_bases_q30_percent": round(100 * after["q30_rate"], 2),
        "illumina_reads_kept_percent": round(
            100 * after["total_reads"] / before["total_reads"], 2),
    })
fastp = pd.DataFrame(fastp_rows)
check("S7", "every released library has a fastp report",
      fastp.illumina_bases.notna().all(),
      sorted(fastp.loc[fastp.illumina_bases.isna(), "library"]))

s7 = s7.merge(fastp, on="library", how="left", validate="1:1")
s7 = s7.merge(s8[["isolate", "total_length_bp"]], on="isolate", how="left", validate="1:1")

s7["ont_reads"] = s7.num_seqs
s7["ont_bases"] = s7.sum_len
s7["ont_mean_read_length_bp"] = s7.avg_len.round(1)
s7["ont_read_n50_bp"] = s7.N50
s7["ont_mean_read_quality"] = s7.AvgQual.round(2)
s7["ont_bases_q20_percent"] = s7["Q20(%)"].round(2)
s7["ont_depth_x"] = (s7.ont_bases / s7.total_length_bp).round(1)
s7["illumina_depth_x"] = (s7.illumina_bases / s7.total_length_bp).round(1)

S7_COLS = {
    "isolate": "Biological sample.",
    "library": "The sequenced preparation the released assembly was built from.",
    "ont_reads": "ONT reads after Filtlong filtering, and after Kraken2 decontamination where it was applied.",
    "ont_bases": "ONT bases in that read set.",
    "ont_mean_read_length_bp": "Mean ONT read length.",
    "ont_read_n50_bp": "ONT read length N50.",
    "ont_mean_read_quality": "Mean ONT per-read Phred quality.",
    "ont_bases_q20_percent": "Percentage of ONT bases at Q20 or better.",
    "ont_depth_x": "ONT bases divided by this isolate's final assembly length from assembly_statistics.tsv.",
    "illumina_read_pairs": "Illumina read pairs surviving fastp.",
    "illumina_bases": "Illumina bases surviving fastp.",
    "illumina_mean_read_length_bp": "Mean Illumina read length after trimming. Not uniform across the collection: most libraries are 2x125 sequencing and four are 2x300 MiSeq runs.",
    "illumina_bases_q30_percent": "Percentage of surviving Illumina bases at Q30 or better.",
    "illumina_reads_kept_percent": "Percentage of raw Illumina reads that survived fastp.",
    "illumina_depth_x": "Illumina bases divided by this isolate's final assembly length from assembly_statistics.tsv.",
    "ont_read_set": "`filtered` for the Filtlong output; `kraken2_decontam_filtered` for the four libraries whose ONT reads had a second organism removed before assembly.",
}
S7 = write_table(s7, "read_statistics", S7_COLS,
                 [SRC["ont_stats_v1"], SRC["ont_stats_v2"], SRC["final_qc"],
                  "02_reads/<library>/qc/<library>_fastp.json"])

[  ok  ] S7     every released library has ONT read statistics


[  ok  ] S7     every released library has a fastp report

Table_S7_per_isolate_read_statistics.tsv — 294 rows × 16 columns from ../pipeline_tables/ont_reads.stats.tsv, ../pipeline_tables/ont_reads.stats_v2.tsv, ../pipeline_tables/final_qc_summary.tsv, 02_reads/<library>/qc/<library>_fastp.json


In [22]:
check("S7", "294 rows", len(S7) == 294, len(S7))
check("S7", "no empty cells", (S7 != "").all().all(),
      {c: int((S7[c] == "").sum()) for c in S7.columns if (S7[c] == "").any()})
decon = set(s7.loc[s7.ont_read_set == "kraken2_decontam_filtered", "library"])
prep = set(final_qc.loc[final_qc.read_prep == "kraken2_decontam", "library"])
check("S7", "ont_read_set == kraken2_decontam_filtered for exactly 4, matching S5 read_prep",
      len(decon) == 4 and decon == prep, f"S7 {sorted(decon)} vs S5 {sorted(prep)}")

EXPECT_S7 = [
    ("ONT reads mean 76,577", round(s7.ont_reads.mean()), 76577),
    ("ONT reads median 66,580", round(s7.ont_reads.median()), 66580),
    ("ONT reads min 4,734", int(s7.ont_reads.min()), 4734),
    ("ONT reads max 567,129", int(s7.ont_reads.max()), 567129),
    ("ONT bases sum 268.8 Gbp", round(s7.ont_bases.sum() / 1e9, 1), 268.8),
    ("ONT bases per-isolate mean 914 Mb", round(s7.ont_bases.mean() / 1e6), 914),
    ("ONT N50 mean 18,215 bp", round(s7.ont_read_n50_bp.mean()), 18215),
    ("ONT mean quality Q16.8", round(s7.ont_mean_read_quality.mean(), 1), 16.8),
    ("ONT bases >= Q20 85.9%", round(s7.ont_bases_q20_percent.mean(), 1), 85.9),
    ("ONT depth mean 165x", round(s7.ont_depth_x.mean()), 165),
    ("ONT depth median 151x", round(s7.ont_depth_x.median()), 151),
    ("ONT depth min 14x", round(s7.ont_depth_x.min()), 14),
    ("ONT depth max 661x", round(s7.ont_depth_x.max()), 661),
    ("Illumina pairs mean 2,742,364", round(s7.illumina_read_pairs.mean()), 2742364),
    ("Illumina pairs median 2,578,010", round(s7.illumina_read_pairs.median()), 2578010),
    ("Illumina pairs min 1,167,252", int(s7.illumina_read_pairs.min()), 1167252),
    ("Illumina pairs max 7,795,268", int(s7.illumina_read_pairs.max()), 7795268),
    ("Illumina bases sum 206.5 Gbp", round(s7.illumina_bases.sum() / 1e9, 1), 206.5),
    ("Illumina bases per-isolate mean 702 Mb", round(s7.illumina_bases.mean() / 1e6), 702),
    ("Illumina Q30 93.1%", round(s7.illumina_bases_q30_percent.mean(), 1), 93.1),
    ("Illumina reads kept 97.4%", round(s7.illumina_reads_kept_percent.mean(), 1), 97.4),
    ("Illumina depth mean 127x", round(s7.illumina_depth_x.mean()), 127),
    ("Illumina depth median 119x", round(s7.illumina_depth_x.median()), 119),
    ("Illumina depth min 56x", round(s7.illumina_depth_x.min()), 56),
    ("Illumina depth max 376x", round(s7.illumina_depth_x.max()), 376),
]
for label, got, want in EXPECT_S7:
    check("S7", label, got == want, f"got {got}, manuscript says {want}")

[  ok  ] S7     294 rows  — 294
[  ok  ] S7     no empty cells
[  ok  ] S7     ont_read_set == kraken2_decontam_filtered for exactly 4, matching S5 read_prep  — S7 ['INF186', 'INF216', 'KSB1_2I', 'KSB2_4A'] vs S5 ['INF186', 'INF216', 'KSB1_2I', 'KSB2_4A']
[  ok  ] S7     ONT reads mean 76,577  — got 76577, manuscript says 76577
[  ok  ] S7     ONT reads median 66,580  — got 66580, manuscript says 66580
[  ok  ] S7     ONT reads min 4,734  — got 4734, manuscript says 4734
[  ok  ] S7     ONT reads max 567,129  — got 567129, manuscript says 567129
[  ok  ] S7     ONT bases sum 268.8 Gbp  — got 268.8, manuscript says 268.8
[  ok  ] S7     ONT bases per-isolate mean 914 Mb  — got 914, manuscript says 914
[  ok  ] S7     ONT N50 mean 18,215 bp  — got 18215, manuscript says 18215
[  ok  ] S7     ONT mean quality Q16.8  — got 16.8, manuscript says 16.8
[  ok  ] S7     ONT bases >= Q20 85.9%  — got 85.9, manuscript says 85.9
[  ok  ] S7     ONT depth mean 165x  — got 165, manuscript says 165
[

## Three-strategy assembly comparison — a working frame, no longer published

The per-isolate comparison that used to ship as Table S10 is not part of the supplement any
more. The Technical Validation paragraph still quotes its aggregates, so the frame is still
built and still checked here — it is simply never written to disk. Delete this section when
that paragraph goes, and the eight rows it backs in `curated/manuscript_expectations.tsv`
with it.

The `ont_*` columns describe **the Nanopore assembly actually carried forward for each
isolate**, not the first-pass Flye run. For four libraries that is the decontaminated
assembly (INF186, INF216, KSB1_2I, KSB2_4A) and for INF205 the Autocycler consensus; all
five are measured from the remediation FASTA, so contig count, length, N50 and largest
contig all come from the same assembly and cannot disagree with each other.

This is what the manuscript aggregate depends on. Remediation improved those five sharply
(INF216 13 → 6 contigs, INF186 6 → 1, KSB1_2I 4 → 1, KSB2_4A 8 → 4, INF205 4 → 2), so the
collection means move from **2.9 contigs / 5.60 Mbp** for the first-pass assemblies to
**2.8 / 5.53** for the ones carried forward. The three-strategy sentence needs the second
pair.

For the seven repolished isolates the hybrid was polished from the `ont` assembly in the
same row, so those two are not independent observations. It does not affect the aggregate,
but it is worth saying in the sentence rather than only here.

In [23]:
asm = pd.read_csv(SRC["asm_stats"], sep="\t")
asm["library"] = asm.file.str.split("/").str[1]
asm["strategy"] = asm.file.str.split("/").str[2]

wide = asm.pivot_table(index="library", columns="strategy",
                       values=["num_seqs", "sum_len", "N50", "max_len"], aggfunc="first")

REMEDIATED = {
    **{p.stem: p for p in Path("06_asm_fixing/02_decontam/assemblies").glob("*.fasta")},
    **{p.stem: p for p in Path("06_asm_fixing/01_autocycler/assemblies").glob("*.fasta")},
}
print(f"remediation assemblies on disk: {sorted(REMEDIATED)}")

strategy_cmp = final_qc[["isolate", "library"]].copy()
strategy_cmp["ont_contigs"] = strategy_cmp.library.map(wide[("num_seqs", "ont_flye")])
strategy_cmp["ont_total_length_bp"] = strategy_cmp.library.map(wide[("sum_len", "ont_flye")])
strategy_cmp["ont_n50_bp"] = strategy_cmp.library.map(wide[("N50", "ont_flye")])
strategy_cmp["ont_largest_contig_bp"] = strategy_cmp.library.map(wide[("max_len", "ont_flye")])
strategy_cmp["illumina_contigs"] = strategy_cmp.library.map(wide[("num_seqs", "illumina_unicycler")])
strategy_cmp["illumina_total_length_bp"] = strategy_cmp.library.map(wide[("sum_len", "illumina_unicycler")])
strategy_cmp["illumina_n50_bp"] = strategy_cmp.library.map(wide[("N50", "illumina_unicycler")])
strategy_cmp["illumina_largest_contig_bp"] = strategy_cmp.library.map(wide[("max_len", "illumina_unicycler")])

# All four ont_* columns are recomputed from the remediation FASTA for the five isolates that
# have one, so they describe a single assembly rather than a mix of two sources.
first_pass = strategy_cmp.set_index("isolate")[["ont_contigs", "ont_total_length_bp"]].copy()
replaced = []
for i, lib in enumerate(strategy_cmp.library):
    path = REMEDIATED.get(lib) or REMEDIATED.get(strategy_cmp.isolate.iloc[i])
    if path is None:
        continue
    lengths = [len(seq) for _, seq in parse_fasta(path)]
    strategy_cmp.loc[strategy_cmp.index[i], ["ont_contigs", "ont_total_length_bp", "ont_n50_bp",
                           "ont_largest_contig_bp"]] = [
        len(lengths), sum(lengths), n50(lengths), max(lengths)]
    replaced.append(strategy_cmp.isolate.iloc[i])

print("ont_* recomputed from the assembly carried forward:")
for iso in sorted(replaced):
    print(f"  {iso:<10} first-pass {first_pass.ont_contigs[iso]:>3} contigs -> "
          f"carried forward {int(strategy_cmp.set_index('isolate').ont_contigs[iso]):>3}")
print(f"\n  mean contigs   first-pass {first_pass.ont_contigs.mean():.3f}  "
      f"carried forward {strategy_cmp.ont_contigs.mean():.3f}")
print(f"  mean total len first-pass {first_pass.ont_total_length_bp.mean() / 1e6:.4f} Mbp  "
      f"carried forward {strategy_cmp.ont_total_length_bp.mean() / 1e6:.4f} Mbp")

strategy_cmp = strategy_cmp.merge(
    s8[["isolate", "contigs", "total_length_bp", "n50_bp"]].rename(columns={
        "contigs": "hybrid_contigs", "total_length_bp": "hybrid_total_length_bp",
        "n50_bp": "hybrid_n50_bp"}), on="isolate", how="left", validate="1:1")
strategy_cmp["hybrid_largest_contig_bp"] = strategy_cmp.isolate.map(
    contig_stats.groupby("isolate").length_bp.max())

for col in ["ont_contigs", "ont_total_length_bp", "ont_n50_bp", "ont_largest_contig_bp",
            "illumina_contigs", "illumina_total_length_bp", "illumina_n50_bp",
            "illumina_largest_contig_bp"]:
    strategy_cmp[col] = strategy_cmp[col].astype("int64")


remediation assemblies on disk: ['INF186', 'INF205', 'INF216', 'KSB1_2I', 'KSB2_4A', 'KSB2_5D_v2']
ont_* recomputed from the assembly carried forward:
  INF186     first-pass   6 contigs -> carried forward   1
  INF205     first-pass   4 contigs -> carried forward   2
  INF216     first-pass  13 contigs -> carried forward   6
  KSB1_2I    first-pass   4 contigs -> carried forward   1
  KSB2_4A    first-pass   8 contigs -> carried forward   4

  mean contigs   first-pass 2.888  carried forward 2.816
  mean total len first-pass 5.6022 Mbp  carried forward 5.5300 Mbp


In [24]:
check("strategy", "294 rows", len(strategy_cmp) == 294, len(strategy_cmp))
check("strategy", "no missing values", strategy_cmp.notna().all().all(),
      {c: int(strategy_cmp[c].isna().sum()) for c in strategy_cmp.columns
       if strategy_cmp[c].isna().any()})
check("strategy", "illumina_contigs mean 112.5 / median 95 / max 390",
      round(strategy_cmp.illumina_contigs.mean(), 1) == 112.5
      and strategy_cmp.illumina_contigs.median() == 95 and strategy_cmp.illumina_contigs.max() == 390,
      f"mean {strategy_cmp.illumina_contigs.mean():.2f} median {strategy_cmp.illumina_contigs.median()} "
      f"max {strategy_cmp.illumina_contigs.max()}")
check("strategy", "ont_contigs mean 2.8", round(strategy_cmp.ont_contigs.mean(), 1) == 2.8,
      f"{strategy_cmp.ont_contigs.mean():.3f}")
check("strategy", "hybrid_contigs mean 2.8", round(strategy_cmp.hybrid_contigs.mean(), 1) == 2.8,
      f"{strategy_cmp.hybrid_contigs.mean():.3f}")
for label, series, want in [
    ("ONT", strategy_cmp.ont_total_length_bp, 5.53),
    ("Illumina", strategy_cmp.illumina_total_length_bp, 5.45),
    ("hybrid", strategy_cmp.hybrid_total_length_bp, 5.53),
]:
    check("strategy", f"{label} mean total length {want} Mbp",
          round(series.mean() / 1e6, 2) == want, f"{series.mean() / 1e6:.4f} Mbp")

# The check that caught the mismatch in the first place: this frame's ONT columns and S5's
# must describe the same assembly for all 294.
by_iso, fq = strategy_cmp.set_index("isolate"), final_qc.set_index("isolate")
check("strategy/S5", "ont_contigs == S5 ont_contigs for all 294",
      by_iso.ont_contigs.astype(int).eq(fq.ont_contigs.astype(int)).all(),
      sorted(by_iso.index[~by_iso.ont_contigs.astype(int).eq(fq.ont_contigs.astype(int))]))
check("strategy/S5", "ont_total_length_bp == S5 ont_total_len for all 294",
      by_iso.ont_total_length_bp.astype(int).eq(fq.ont_total_len.astype(int)).all(),
      sorted(by_iso.index[~by_iso.ont_total_length_bp.astype(int).eq(fq.ont_total_len.astype(int))]))
check("strategy", "the 5 recomputed isolates are the decontaminated four plus INF205",
      set(replaced) == {"INF186", "INF216", "KSB1_2I", "KSB2_4A", "INF205"},
      sorted(replaced))
check("strategy/S8", "hybrid columns agree with S8",
      strategy_cmp.set_index("isolate").hybrid_total_length_bp.eq(
          s8.set_index("isolate").total_length_bp).all())

[  ok  ] strategy 294 rows  — 294
[  ok  ] strategy no missing values
[  ok  ] strategy illumina_contigs mean 112.5 / median 95 / max 390  — mean 112.52 median 95.0 max 390
[  ok  ] strategy ont_contigs mean 2.8  — 2.816
[  ok  ] strategy hybrid_contigs mean 2.8  — 2.847
[  ok  ] strategy ONT mean total length 5.53 Mbp  — 5.5300 Mbp
[  ok  ] strategy Illumina mean total length 5.45 Mbp  — 5.4489 Mbp
[  ok  ] strategy hybrid mean total length 5.53 Mbp  — 5.5295 Mbp
[  ok  ] strategy/S5 ont_contigs == S5 ont_contigs for all 294
[  ok  ] strategy/S5 ont_total_length_bp == S5 ont_total_len for all 294
[  ok  ] strategy the 5 recomputed isolates are the decontaminated four plus INF205  — ['INF186', 'INF205', 'INF216', 'KSB1_2I', 'KSB2_4A']
[  ok  ] strategy/S8 hybrid columns agree with S8


True

## Table 1 — Composition and assembly statistics per species

Grouped from S8 and S6 rather than read from `species_table.tsv`, so Table 1 and the
supplement cannot disagree.

The `All` row is **not a column sum** for the diversity columns: STs and loci are shared
between species, so those cells are distinct counts over the whole collection. That needs a
footnote in the manuscript.

Single-contig counts per species are available and deliberately not shown. 234 isolates carry
plasmids, so a single-contig genome usually means an isolate with nothing extrachromosomal,
not a better assembly.

In [25]:
t1_src = s8.merge(s6[["isolate", "hybrid_species", "hybrid_ST", "hybrid_K_locus"]],
                   on="isolate", validate="1:1")


def species_row(label, grp):
    return {
        "species": label,
        "n_isolates": len(grp),
        "n_distinct_ST": grp.hybrid_ST.nunique(),
        "n_distinct_K_locus": grp.hybrid_K_locus.nunique(),
        "n_closed_chromosome": int((grp.chromosome_circular == "yes").sum()),
        "percent_closed_chromosome": round(
            100 * (grp.chromosome_circular == "yes").mean(), 1),
        "n_fully_circular": int((grp.all_contigs_circular == "yes").sum()),
        "percent_fully_circular": round(
            100 * (grp.all_contigs_circular == "yes").mean(), 1),
        "median_contigs": grp.contigs.median(),
        "median_total_length_bp": int(grp.total_length_bp.median()),
        "median_chromosome_length_bp": int(grp.chromosome_length_bp.median()),
    }


t1 = pd.DataFrame(
    [species_row(sp, grp) for sp, grp in t1_src.groupby("hybrid_species")]
    + [species_row("All", t1_src)]
)

T1_COLS = {
    "species": "Species or subspecies as called by Kleborate on the released hybrid assembly. The `All` row covers the whole collection.",
    "n_isolates": "Released genomes.",
    "n_distinct_ST": "Distinct sequence types. The `All` row is a distinct count over the collection, not a column sum: STs are shared between species.",
    "n_distinct_K_locus": "Distinct K loci. The `All` row is a distinct count, not a column sum.",
    "n_closed_chromosome": "Genomes whose chromosome is flagged circular. This is the completeness figure the manuscript quotes.",
    "percent_closed_chromosome": "n_closed_chromosome as a percentage of n_isolates.",
    "n_fully_circular": "Genomes in which every contig is flagged circular.",
    "percent_fully_circular": "n_fully_circular as a percentage of n_isolates.",
    "median_contigs": "Median contigs per assembly.",
    "median_total_length_bp": "Median total assembly length.",
    "median_chromosome_length_bp": "Median chromosome length.",
}
T1 = write_table(t1, "Table_1_species_composition", T1_COLS,
                 [SRC["final_qc"], SRC["assemblies"]], sort_by=[])


Table_1_species_composition.tsv — 5 rows × 11 columns from ../pipeline_tables/final_qc_summary.tsv, 10_final_assemblies/assemblies


In [26]:
check("T1", "5 rows", len(T1) == 5, len(T1))
body = t1[t1.species != "All"]
allrow = t1[t1.species == "All"].iloc[0]
check("T1", "n_isolates sums to 294", body.n_isolates.sum() == 294, body.n_isolates.sum())
check("T1", "per-species n_isolates are 224 / 56 / 11 / 3",
      sorted(body.n_isolates, reverse=True) == [224, 56, 11, 3],
      sorted(body.n_isolates, reverse=True))
check("T1", "All row reads 294 / 212 / 96 / 276 (93.9%) / 224 (76.2%)",
      (allrow.n_isolates, allrow.n_distinct_ST, allrow.n_distinct_K_locus,
       allrow.n_closed_chromosome, allrow.percent_closed_chromosome,
       allrow.n_fully_circular, allrow.percent_fully_circular)
      == (294, 212, 96, 276, 93.9, 224, 76.2),
      allrow.to_dict())
check("T1", "per-species closed-chromosome counts sum to 276",
      body.n_closed_chromosome.sum() == 276, body.n_closed_chromosome.sum())
check("T1", "per-species fully-circular counts sum to 224",
      body.n_fully_circular.sum() == 224, body.n_fully_circular.sum())
# K loci are shared between species, so the All row is genuinely not a column sum. STs are
# not shared here — 155 + 46 + 8 + 3 = 212 — which is a fact about this collection, not a
# guarantee, so the footnote should be written about the K-locus column.
check("T1", "All row K-locus count is a distinct count, not a column sum",
      allrow.n_distinct_K_locus < body.n_distinct_K_locus.sum(),
      f"{allrow.n_distinct_K_locus} distinct vs {body.n_distinct_K_locus.sum()} summed")
check("T1", "no ST is shared between species in this collection, so the ST column happens to sum",
      allrow.n_distinct_ST == body.n_distinct_ST.sum(),
      f"{allrow.n_distinct_ST} distinct vs {body.n_distinct_ST.sum()} summed")
T1

[  ok  ] T1     5 rows  — 5
[  ok  ] T1     n_isolates sums to 294  — 294
[  ok  ] T1     per-species n_isolates are 224 / 56 / 11 / 3  — [224, 56, 11, 3]
[  ok  ] T1     All row reads 294 / 212 / 96 / 276 (93.9%) / 224 (76.2%)  — {'species': 'All', 'n_isolates': 294, 'n_distinct_ST': 212, 'n_distinct_K_locus': 96, 'n_closed_chromosome': 276, 'percent_closed_chromosome': 93.9, 'n_fully_circular': 224, 'percent_fully_circular': 76.2, 'median_contigs': 2.0, 'median_total_length_bp': 5519564, 'median_chromosome_length_bp': 5307218}
[  ok  ] T1     per-species closed-chromosome counts sum to 276  — 276
[  ok  ] T1     per-species fully-circular counts sum to 224  — 224
[  ok  ] T1     All row K-locus count is a distinct count, not a column sum  — 96 distinct vs 121 summed
[  ok  ] T1     no ST is shared between species in this collection, so the ST column happens to sum  — 212 distinct vs 212 summed


,species,n_isolates,n_distinct_ST,n_distinct_K_locus,n_closed_chromosome,percent_closed_chromosome,n_fully_circular,percent_fully_circular,median_contigs,median_total_length_bp,median_chromosome_length_bp
0,Klebsiella pneumoniae,224,155,80,211,94.2,172,76.8,2.0,5499944,5282955
1,Klebsiella quasipneumoniae subsp. quasipneumoniae,3,3,2,3,100.0,3,100.0,1.0,5295856,5295856
2,Klebsiella quasipneumoniae subsp. similipneumo...,11,8,8,9,81.8,7,63.6,3.0,5432257,5141367
3,Klebsiella variicola subsp. variicola,56,46,31,53,94.6,42,75.0,2.0,5700309,5496302
4,All,294,212,96,276,93.9,224,76.2,2.0,5519564,5307218


## Cross-table integrity checks

Run after every rebuild. Each one is a claim that two tables describe the same collection.

In [27]:
tables = {"S1": S1, "S2": S2, "S3": S3, "S4": S4, "S5": S5, "S6": S6, "S7": S7,
          "S8": S8, "T1": T1}

released_isolates = set(S2.loc[S2.released == "yes", "isolate"])
for name, tbl in [("S6", S6), ("S7", S7), ("S8", S8), ("S5", S5[S5.released == "yes"])]:
    check("cross", f"{name} covers exactly the 294 released isolates",
          set(tbl.isolate) == released_isolates and len(tbl) == len(released_isolates),
          f"{len(set(tbl.isolate))} isolates, {len(tbl)} rows")

check("cross", "S8 total_length_bp == S5 hybrid_total_len",
      S8.set_index("isolate").total_length_bp.astype(int).eq(
          S5[S5.released == "yes"].set_index("isolate").hybrid_total_len.astype(int)).all())
check("cross", "S8 chromosome_length_bp == S5 hybrid_chromosome_len",
      S8.set_index("isolate").chromosome_length_bp.astype(int).eq(
          S5[S5.released == "yes"].set_index("isolate").hybrid_chromosome_len.astype(int)).all())
check("cross", "S8 contigs == the per-contig frame's rows per isolate",
      S8.set_index("isolate").contigs.astype(int).eq(
          contig_stats.groupby("isolate").size()).all())
check("cross", "S6 distinct ST and K-locus counts == the T1 All row",
      S6.hybrid_ST.nunique() == int(allrow.n_distinct_ST)
      and S6.hybrid_K_locus.nunique() == int(allrow.n_distinct_K_locus),
      f"{S6.hybrid_ST.nunique()} / {S6.hybrid_K_locus.nunique()} vs "
      f"{allrow.n_distinct_ST} / {allrow.n_distinct_K_locus}")
check("cross", "T1 per-species counts == S8 grouped on species",
      body.n_closed_chromosome.sum() == (S8.chromosome_circular == "yes").sum())
check("cross", "S7 decontaminated set == S5 kraken2_decontam set == 4 isolates",
      set(S7.loc[S7.ont_read_set == "kraken2_decontam_filtered", "isolate"])
      == set(S5.loc[S5.read_prep == "kraken2_decontam", "isolate"])
      == set(S8.loc[S8.assembly_route == "kraken2 decontam > clinopore", "isolate"]))
check("cross", "S4's 28 libraries are all in S2's 316", set(S4.library) <= set(S2.library))
prov = S4.loc[S4.verdict == "PASS_PROVISIONAL_CHECK_EXTERNAL", "library"]
ext_of_prov = S5.set_index("library").loc[list(prov), "external_qc_decision"]
check("cross", "S4's four provisional passes appear in S5 with external_qc_decision == PASS",
      set(ext_of_prov) == {"PASS"}, ext_of_prov.to_dict())
check("cross", "S1 run IDs cover every run token in S2",
      {t.split(":")[0] for s in S2.ont_run_barcode if s for t in s.split(";")} <= set(S1.run_id))

# quality_control_and_routing.tsv carries the internal-QC typing for all 316 libraries; genome_typing.tsv carries the
# typing of the assembly each released genome was finally built from. Same call for 292 of
# the 294 released libraries; the two exceptions are the libraries reassembled from
# decontaminated reads. Naming them means a third cannot appear unnoticed.
rel5_typing = S5[S5.released == "yes"].set_index("isolate")
s6_typing = S6.set_index("isolate")
for col in ("ont_species", "ont_ST", "ont_K_locus"):
    differ = sorted(rel5_typing.index[
        ~rel5_typing[col].eq(s6_typing[col].reindex(rel5_typing.index))])
    check("cross", f"S5 and S6 {col} agree except for INF186 and INF216",
          set(differ) <= {"INF186", "INF216"}, differ)
for col in ("illumina_species", "illumina_ST", "illumina_K_locus"):
    differ = sorted(rel5_typing.index[
        ~rel5_typing[col].eq(s6_typing[col].reindex(rel5_typing.index))])
    check("cross", f"S5 and S6 {col} agree for all 294 released isolates", not differ, differ)

check("cross", "S4 and S5 agree on mapped_pct and kraken2_status for the 28 adjudicated",
      S4.set_index("library")[["mapped_pct", "kraken2_status"]].sort_index().equals(
          S5.set_index("library").loc[S4.library, ["mapped_pct", "kraken2_status"]].sort_index()))

# No table may ship a placeholder.
placeholders = {name: [c for c in tbl.columns if (tbl[c] == "confirm").any()]
                for name, tbl in tables.items()}
placeholders = {k: v for k, v in placeholders.items() if v}
check("cross", "no cell in any table equals the literal string `confirm`",
      not placeholders, placeholders)

# Every tool named in Methods must appear in S3 exactly once. The list is the tools the
# Methods text names; add to it when Methods does.
METHODS_TOOLS = [
    "Dorado", "Filtlong", "fastp", "Flye", "Unicycler", "minimap2", "samtools", "Kraken2",
    "KrakenTools", "Kleborate", "Autocycler", "Medaka", "Polypolish", "POLCA", "Nextflow",
    "seqkit",
]
tool_counts = {t: int((S3.tool == t).sum()) for t in METHODS_TOOLS}
check("cross", "every tool named in Methods appears exactly once in S3",
      set(tool_counts.values()) == {1}, {t: n for t, n in tool_counts.items() if n != 1})

[  ok  ] cross  S6 covers exactly the 294 released isolates  — 294 isolates, 294 rows
[  ok  ] cross  S7 covers exactly the 294 released isolates  — 294 isolates, 294 rows
[  ok  ] cross  S8 covers exactly the 294 released isolates  — 294 isolates, 294 rows
[  ok  ] cross  S5 covers exactly the 294 released isolates  — 294 isolates, 294 rows
[  ok  ] cross  S8 total_length_bp == S5 hybrid_total_len
[  ok  ] cross  S8 chromosome_length_bp == S5 hybrid_chromosome_len
[  ok  ] cross  S8 contigs == the per-contig frame's rows per isolate
[  ok  ] cross  S6 distinct ST and K-locus counts == the T1 All row  — 212 / 96 vs 212 / 96
[  ok  ] cross  T1 per-species counts == S8 grouped on species
[  ok  ] cross  S7 decontaminated set == S5 kraken2_decontam set == 4 isolates
[  ok  ] cross  S4's 28 libraries are all in S2's 316
[  ok  ] cross  S4's four provisional passes appear in S5 with external_qc_decision == PASS  — {'INF195': 'PASS', 'KSB1_2F': 'PASS', 'KSB1_3I': 'PASS', 'KSB1_6D_v2': 'PASS'

True

## Manuscript-number check

Every number quoted in the draft, recomputed from the tables above. The expectations live in
`curated/manuscript_expectations.tsv` with a column saying where in the manuscript each one
appears, so changing a number in the paper means changing one line in a file that then
verifies itself.

Where the same value is quoted in two places it appears twice, because the two sentences can
drift apart independently.

In [28]:
expect = pd.read_csv(SRC["cur_expect"], sep="\t", comment="#")
namespace = {"pd": pd, **{k: v for k, v in tables.items()}}
# the checks want numeric frames, not the rendered strings
numeric = {}
for name, df in [("S1", s1), ("S2", s2), ("S3", s3_software), ("S4", s4), ("S5", s5),
                 ("S6", s6), ("S7", s7), ("S8", s8), ("T1", t1),
                 ("contig_stats", contig_stats), ("strategy_cmp", strategy_cmp)]:
    numeric[name] = df
namespace.update(numeric)

results = []
for row in expect.itertuples():
    try:
        got = eval(row.expression, {"pd": pd}, namespace)
        got = float(got)
        ok = abs(got - float(row.quoted_value)) <= float(row.tolerance)
        detail = f"{got:,.4g}"
    except Exception as exc:
        ok, detail = False, f"error: {exc}"
    results.append({
        "quoted_value": row.quoted_value,
        "recomputed": detail,
        "ok": ok,
        "where_in_manuscript": row.where_in_manuscript,
        "source_table": row.source_table,
        "derivation": row.derivation,
    })

manuscript = pd.DataFrame(results)
failed = manuscript[~manuscript.ok]
check("manuscript", f"all {len(manuscript)} quoted numbers reproduce",
      failed.empty, f"{len(failed)} do not")

if not failed.empty:
    print("\nNUMBERS THAT DO NOT REPRODUCE — the manuscript sentence is what changes:\n")
    print(failed[["quoted_value", "recomputed", "where_in_manuscript", "derivation"]]
          .to_string(index=False))

[  ok  ] manuscript all 81 quoted numbers reproduce  — 0 do not


## Report

In [29]:
stale = [p.name for p in sorted(OUT.glob("*.tsv")) if p.name not in WRITTEN]
check("output", "nothing stale left in 11_manuscript/supplementary", not stale, stale)

report = pd.DataFrame(CHECKS)
failures = report[~report.ok]
print(f"{len(report)} assertions · {len(report) - len(failures)} passed · {len(failures)} failed\n")
if not failures.empty:
    print(failures[["table", "check", "detail"]].to_string(index=False))
else:
    print("Every assertion passed.")

print("\nfiles written:")
for name in sorted(WRITTEN):
    path = OUT / name
    print(f"  {path.relative_to(ROOT)}  ({path.stat().st_size:,} bytes)")
if stale:
    print("\nSTALE — in the output directory but not written by this run; delete them:")
    for name in stale:
        print(f"  {(OUT / name).relative_to(ROOT)}")

[  ok  ] output nothing stale left in 11_manuscript/supplementary
164 assertions · 164 passed · 0 failed

Every assertion passed.

files written:
  11_manuscript/supplementary/Table_1_species_composition.tsv  (595 bytes)
  11_manuscript/supplementary/Table_S1_ont_run_metadata.tsv  (3,355 bytes)
  11_manuscript/supplementary/Table_S2_sample_manifest.tsv  (25,154 bytes)
  11_manuscript/supplementary/Table_S3_software_versions.tsv  (1,581 bytes)
  11_manuscript/supplementary/Table_S4_internal_qc_adjudications.tsv  (5,852 bytes)
  11_manuscript/supplementary/Table_S5_qc_and_routing.tsv  (71,636 bytes)
  11_manuscript/supplementary/Table_S6_typing.tsv  (56,995 bytes)
  11_manuscript/supplementary/Table_S7_per_isolate_read_statistics.tsv  (32,297 bytes)
  11_manuscript/supplementary/Table_S8_per_assembly_statistics.tsv  (26,638 bytes)
